# EDA-1 — Bằng chứng cho Transaction Construction

So sánh patch **4×4, stride 4 (49 patch/ảnh)** với **7×7, stride 7 (16 patch/ảnh)** trên 70.000 ảnh gộp. Hai cách cắt đều giữ đủ 784 pixel trước lọc. Notebook trả lời 15 câu hỏi I1–I5; không tạo codebook, chọn K/minsup hoặc tạo transaction chính thức.

Phần D–A–B chuẩn bị nguồn và hai ngưỡng; phần I1–I5 dùng lại dữ liệu để đánh giá cấu trúc cục bộ và mất thông tin sau lọc. SIFT/ORB và descriptor diversity chỉ dùng pilot 200 ảnh, 20/lớp, seed 42. Nhãn phục vụ phân tích, không đưa vào item.

**Cách chạy:** chọn kernel Julia của repo, restart kernel rồi chạy từ đầu. Lần refactor này chưa chạy toàn bộ notebook; output cũ đã được bỏ khỏi các cell để tránh nhầm bằng chứng.

## Quy ước và môi trường

EDA-1 dùng toàn bộ 70.000 ảnh gộp. Nhãn gốc là 0–9; chỉ số mảng lớp là nhãn + 1. Output lưu ở results/fashion/eda1/.

### SET01 — Repo và Julia project

- **Câu hỏi:** Notebook đã kích hoạt đúng project chưa?
- **Cách tính:** Tìm repo root, kích hoạt Julia project và nạp dependencies.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
using Pkg

"""Tìm repo từ thư mục hiện tại trở lên; trả về đường dẫn hoặc báo lỗi khi không tìm thấy."""
function find_repo_root(start=pwd())
    directory = abspath(start)
    while true
        if isfile(joinpath(directory, "Project.toml")) && isfile(joinpath(directory, "docs", "plan.md")) && isdir(joinpath(directory, "src"))
            return directory
        end
        parent = dirname(directory)
        parent == directory && error("Không tìm thấy repo root.")
        directory = parent
    end
end

repo_root = find_repo_root()
Pkg.activate(repo_root)
Pkg.instantiate()

using EclatFIM, PNGFiles, CodecZlib
using ColorTypes: Gray
using FixedPointNumbers: N0f8
using SHA, TOML, Random, OpenCV, Printf, Markdown

@assert VERSION >= v"1.9"

### SET02 — Seed, tham số và đường dẫn

- **Câu hỏi:** Tham số nào điều khiển khả năng tái lập?
- **Cách tính:** Khai báo seed, kích thước dữ liệu, cấu hình và thư mục input/output.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
const SEED = 42
const N = 70_000
const H = 28
const W = 28
const NPX = H * W
const PATCH_SIZES = (4, 7)
const ALPHAS = (0.0, 1 / 16, 2 / 16)
const EDGE_TAU = 80.0
const NBINS = 8
const PILOT_PER_CLASS = 20
const CLASSES = EclatFIM.FASHION_CLASSES

const DATA = joinpath(repo_root, "data", "processed", "fashion-mnist", "combined")
const CACHE = joinpath(repo_root, "data", "processed", "fashion-mnist", "eda1")
const OUT = joinpath(repo_root, "results", "fashion", "eda1", "question_driven")
mkpath(OUT)
mkpath(CACHE)

println("Julia ", VERSION, " | OpenCV.jl ", Base.pkgversion(OpenCV), " | OpenCV ", OpenCV.getVersionString())
println("Đầu vào: ", relpath(DATA, repo_root))
println("Output: ", relpath(OUT, repo_root))
println("Patch 4×4: 49 ô; patch 7×7: 16 ô; α = ", ALPHAS)

### UTIL01 — CSV

Cell chỉ định nghĩa các hàm dùng chung.

In [ ]:
struct SVGFigure
    source::String
end
Base.show(io::IO,::MIME"image/svg+xml",x::SVGFigure)=print(io,x.source)
Base.show(io::IO,::MIME"text/plain",::SVGFigure)=print(io,"SVG saved under results/fashion/eda1/")
function csv_cell(x)
    s=string(x)
    occursin(',',s)||occursin('"',s)||occursin('\n',s) ? "\""*replace(s,"\""=>"\"\"")*"\"" : s
end
function write_csv(path,header,rows)
    open(path,"w") do io
        println(io,join(csv_cell.(header),","))
        for row in rows println(io,join(csv_cell.(row),",")) end
    end
    path
end


### UTIL02 — SVG cơ bản

Cell chỉ định nghĩa các hàm dùng chung.

In [ ]:
function svg_text(io,x,y,s;size=12,anchor="start",color="#172338",weight="normal")
    t=replace(string(s),"&"=>"&amp;","<"=>"&lt;",">"=>"&gt;","\""=>"&quot;")
    println(io,"<text x=\"$x\" y=\"$y\" font-family=\"sans-serif\" font-size=\"$size\" font-weight=\"$weight\" text-anchor=\"$anchor\" fill=\"$color\">$t</text>")
end
function svg_save(path,title,body,w,h)
    io=IOBuffer();println(io,"<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"$w\" height=\"$h\" viewBox=\"0 0 $w $h\"><rect width=\"100%\" height=\"100%\" fill=\"white\"/>")
    svg_text(io,w/2,28,title;size=18,anchor="middle",weight="bold");print(io,body,"</svg>")
    s=String(take!(io));write(path,s);SVGFigure(s)
end


### UTIL03 — Biểu đồ

Cell chỉ định nghĩa các hàm dùng chung.

In [ ]:
function bar_plot(labels,values,title,path;ylabel="",color="#3478b8")
    w=1000;h=490;l=75;t=65;pw=900;ph=330;ym=max(maximum(values),1.0);io=IOBuffer()
    for k in 0:4
        yy=t+ph*k/4;svg_text(io,l-8,yy+4,@sprintf("%.3g",ym*(4-k)/4);size=10,anchor="end",color="#596579")
        println(io,"<line x1=\"$l\" y1=\"$yy\" x2=\"$(l+pw)\" y2=\"$yy\" stroke=\"#e7ebf0\"/>")
    end
    bw=pw/length(values)*0.7
    for i in eachindex(values)
        x=l+pw*(i-0.5)/length(values);bh=ph*values[i]/ym
        println(io,"<rect x=\"$(x-bw/2)\" y=\"$(t+ph-bh)\" width=\"$bw\" height=\"$bh\" fill=\"$color\"/>")
        svg_text(io,x,t+ph+18,labels[i];size=10,anchor="middle")
    end
    svg_text(io,l+pw/2,h-15,ylabel;anchor="middle")
    svg_save(path,title,String(take!(io)),w,h)
end
"""Vẽ nhiều chuỗi cột chồng nhau; chuỗi đầu tiên nằm phía trước."""
function overlaid_bar_plot(labels, values, series_labels, title, path; ylabel="", xlabel="", colors=["#d97706", "#147d64"])
    w = 1000
    h = 490
    left = 75
    top = 65
    plot_width = 760
    plot_height = 330
    y_max = maximum(maximum(series) for series in values)
    y_max = max(y_max, 1.0)
    io = IOBuffer()

    for tick in 0:4
        y = top + plot_height * tick / 4
        value = y_max * (4 - tick) / 4
        svg_text(io, left - 8, y + 4, @sprintf("%.2g", value); size=10, anchor="end", color="#596579")
        println(io, "<line x1=\"$left\" y1=\"$y\" x2=\"$(left + plot_width)\" y2=\"$y\" stroke=\"#e7ebf0\"/>")
    end

    group_width = plot_width / length(labels)
    background_width = group_width * 0.68
    foreground_width = group_width * 0.42

    # Vẽ chuỗi sau trước để chuỗi đầu tiên nổi lên phía trước.
    for series_index in length(values):-1:1
        series = values[series_index]
        # bar_width = series_index == 1 ? foreground_width : background_width
        # opacity = series_index == 1 ? 1.0 : 0.42
        bar_width = background_width
        opacity = 1

        for index in eachindex(labels)
            center = left + group_width * (index - 0.5)
            bar_height = plot_height * series[index] / y_max
            y = top + plot_height - bar_height
            println(io, "<rect x=\"$(center - bar_width / 2)\" y=\"$y\" width=\"$bar_width\" height=\"$bar_height\" fill=\"$(colors[series_index])\" fill-opacity=\"$opacity\"/>")
        end
    end

    for index in eachindex(labels)
        center = left + group_width * (index - 0.5)
        svg_text(io, center, top + plot_height + 18, labels[index]; size=10, anchor="middle")
    end

    for series_index in eachindex(series_labels)
        y = top + 18 * series_index
        println(io, "<rect x=\"$(left + plot_width + 24)\" y=\"$(y - 9)\" width=\"12\" height=\"12\" fill=\"$(colors[series_index])\"/>")
        svg_text(io, left + plot_width + 42, y + 1, series_labels[series_index]; size=10)
    end

    println(io, "<g transform=\"translate(20,$(top + plot_height / 2)) rotate(-90)\">")
    svg_text(io, 0, 0, ylabel; anchor="middle")
    println(io, "</g>")
    svg_text(io, left + plot_width / 2, h - 15, xlabel; anchor="middle")
    svg_save(path, title, String(take!(io)), w, h)
end

function line_plot(series,title,path;ylabel="",xmin=0,xmax=255,ymax=nothing,markers=Tuple{Float64,String,String}[])
    w=1120;h=530;l=75;t=60;pw=760;ph=350
    ym=isnothing(ymax) ? maximum(maximum(s.values) for s in series) : ymax;ym=max(ym,eps());io=IOBuffer()
    for k in 0:4
        yy=t+ph*k/4;xx=l+pw*k/4
        println(io,"<line x1=\"$l\" y1=\"$yy\" x2=\"$(l+pw)\" y2=\"$yy\" stroke=\"#e7ebf0\"/>")
        svg_text(io,l-8,yy+4,@sprintf("%.3g",ym*(4-k)/4);size=10,anchor="end",color="#596579")
        svg_text(io,xx,t+ph+18,@sprintf("%.3g",xmin+(xmax-xmin)*k/4);size=10,anchor="middle")
    end
    for (j,s) in enumerate(series)
        pts=String[]
        for (i,v) in enumerate(s.values)
            x=l+pw*(i-1)/max(length(s.values)-1,1);y=t+ph*(1-v/ym)
            push!(pts,@sprintf("%.2f,%.2f",x,y))
        end
        println(io,"<polyline fill=\"none\" stroke=\"$(s.color)\" stroke-width=\"2\" points=\"$(join(pts," "))\"/>")
        yy=t+18*j;println(io,"<line x1=\"$(l+pw+20)\" y1=\"$yy\" x2=\"$(l+pw+44)\" y2=\"$yy\" stroke=\"$(s.color)\" stroke-width=\"3\"/>")
        svg_text(io,l+pw+52,yy+4,s.label;size=10)
    end
    for (xvalue,label,color) in markers
        x=l+pw*(xvalue-xmin)/max(xmax-xmin,eps())
        println(io, "<line x1=\"$x\" y1=\"$t\" x2=\"$x\" y2=\"$(t+ph)\" stroke=\"$color\" stroke-dasharray=\"5,4\" stroke-width=\"2\"/>")
        svg_text(io,x+4,t+14,label;size=9,color)
    end
    svg_text(io,l+pw/2,h-14,"Cường độ pixel (0–255)";anchor="middle")
    svg_save(path,title,String(take!(io)),w,h)
end

function line_plot_with_tau_markers(series, title, path; ylabel="", xmin=0, xmax=255, ymax=nothing, markers=[])
    figure = line_plot(series, title, path; ylabel=ylabel, xmin=xmin, xmax=xmax, ymax=ymax)
    marker_io = IOBuffer()
    left = 75
    top = 60
    plot_width = 760
    plot_height = 350

    for (tau, label, color) in markers
        x = left + plot_width * (tau - xmin) / max(xmax - xmin, eps())
        println(marker_io, "<line x1=\"$x\" y1=\"$top\" x2=\"$x\" y2=\"$(top + plot_height)\" stroke=\"$color\" stroke-dasharray=\"5,4\" stroke-width=\"2\"/>")
        svg_text(marker_io, x + 4, top + 14, label; size=9, color=color)
    end

    marked_svg = replace(figure.source, "</svg>" => String(take!(marker_io)) * "</svg>")
    write(path, marked_svg)
    SVGFigure(marked_svg)
end

function draw_image(io,img,x,y;scale=5,binary=false)
    for r in axes(img,1),c in axes(img,2)
        v=binary ? (img[r,c] ? 255 : 0) : Int(img[r,c])
        println(io,"<rect x=\"$(x+(c-1)*scale)\" y=\"$(y+(r-1)*scale)\" width=\"$scale\" height=\"$scale\" fill=\"rgb($v,$v,$v)\"/>")
    end
end


### UTIL04 — Thống kê

Cell chỉ định nghĩa các hàm dùng chung.

In [ ]:
function hist_quantile(hist,rank)
    s=0
    for i in eachindex(hist) s+=hist[i];s>=rank&&return i-1 end
    length(hist)-1
end
function qlinear(v,p)
    a=sort!(Float64.(collect(v)));isempty(a)&&return NaN
    x=1+(length(a)-1)*p;i=floor(Int,x);j=ceil(Int,x);a[i]+(x-i)*(a[j]-a[i])
end
function summary5(v)
    (length(v),sum(v)/length(v),qlinear(v,.5),qlinear(v,.05),qlinear(v,.95),minimum(v),maximum(v))
end

function mean_value(values)
    sum(values) / length(values)
end


nothing

## 1. D — Tổng quan và xác minh dữ liệu

Nạp index theo image_id, giữ provenance và xác minh checksum.

### D01 — Metadata và index

- **Câu hỏi:** Index có đúng checksum, header và số dòng?
- **Cách tính:** Đối chiếu metadata trước khi đọc ảnh.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
meta=TOML.parsefile(joinpath(DATA,"metadata.toml"));idxpath=joinpath(DATA,"image_index.csv")
idxsha=bytes2hex(open(sha256,idxpath))
@assert idxsha==meta["sha256"]["image_index.csv"]&&meta["images"]==N
lines=readlines(idxpath);@assert length(lines)==N+1
@assert lines[1]=="image_id,relative_path,label,source_split,source_index,sha256"


### D02 — Nạp ảnh PNG theo image_id

- **Câu hỏi:** Ảnh, nhãn và split có khớp từng dòng manifest?
- **Cách tính:** Đọc PNG theo image_id, kiểm tra SHA-256, kích thước và kiểu pixel.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
X=Array{UInt8}(undef,28,28,N);y=zeros(Int,N);split_name=Vector{String}(undef,N);source_idx=zeros(Int,N)
pixel_groups=Dict{String,Vector{Int}}();black_ids=Int[];white_ids=Int[];lo,hi=typemax(UInt8),typemin(UInt8)
for id in 1:N
    f=split(lines[id+1],',');@assert length(f)==6&&parse(Int,f[1])==id
    y[id]=parse(Int,f[3]);split_name[id]=f[4];source_idx[id]=parse(Int,f[5])
    @assert split_name[id]==(id<=60_000 ? "train" : "test")
    @assert source_idx[id]==(id<=60_000 ? id : id-60_000)&&0<=y[id]<=9
    p=joinpath(DATA,f[2]);@assert bytes2hex(open(sha256,p))==f[6]
    im=PNGFiles.load(p);@assert size(im)==(28,28)&&eltype(im)==Gray{N0f8}
    pix=Matrix{UInt8}(reinterpret(UInt8,im));X[:,:,id]=pix
    lo=min(lo,minimum(pix));hi=max(hi,maximum(pix))
    all(iszero,pix)&&push!(black_ids,id);all(==(255),pix)&&push!(white_ids,id)
    key=bytes2hex(sha256(vec(pix)));push!(get!(pixel_groups,key,Int[]),id)
    id%10_000==0&&println("Verified PNGs $id / $N")
end


### D03–D04 — Split, nhãn, ảnh trùng và ngoại lệ

- **Câu hỏi:** Có đủ mẫu và ảnh nào toàn đen/trắng hoặc trùng pixel?
- **Cách tính:** Kiểm tra split, cân bằng lớp, duplicate candidates và ảnh biên.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
@assert size(X)==(28,28,70_000)&&eltype(X)==UInt8
@assert split_name[60_000]=="train"&&split_name[60_001]=="test"
class_counts=[count(==(k),y) for k in 0:9];@assert class_counts==fill(7_000,10)
dups=[g for g in values(pixel_groups) if length(g)>1];dup_rows=Any[]
for g in dups
    for id in g[2:end] @assert X[:,:,id]==X[:,:,first(g)] end
    for id in g push!(dup_rows,(id,y[id],split_name[id],source_idx[id],first(g),length(g),length(unique(y[g]))>1,length(unique(split_name[g]))>1)) end
end


### D05 — Báo cáo chất lượng

- **Câu hỏi:** Chất lượng toàn tập được tóm tắt ra sao?
- **Cách tính:** Xuất metadata chất lượng và bảng thành viên nhóm ảnh trùng.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
write_csv(joinpath(OUT,"01_duplicate_members.csv"),
    ["image_id","label","source_split","source_index","group_reference","group_size","cross_label","cross_split"],dup_rows)
write_csv(joinpath(OUT,"01_dataset_overview.csv"),["metric","value"],[
    ("images",N),("classes",10),("train_source",60_000),("test_source",10_000),("images_per_class",7_000),
    ("shape","28x28x70000"),("pixel_type","UInt8"),("pixel_min",Int(lo)),("pixel_max",Int(hi)),
    ("pixel_count",length(X)),("all_black_images",length(black_ids)),("all_white_images",length(white_ids)),
    ("duplicate_groups",length(dups)),("extra_duplicate_copies",sum(length(g)-1 for g in dups;init=0)),("index_sha256",idxsha)])


### D06 — Số ảnh theo lớp

- **Câu hỏi:** Tập gộp có cân bằng nhãn?
- **Cách tính:** Đếm ảnh và tỷ lệ theo nhãn 0–9.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
write_csv(joinpath(OUT,"01_class_counts.csv"),["label","class_name","images","fraction"],
    [(k,CLASSES[k+1],class_counts[k+1],class_counts[k+1]/N) for k in 0:9])


### D07 — Biểu đồ lớp

- **Câu hỏi:** Phân bố ảnh theo lớp nhìn ra sao?
- **Cách tính:** Vẽ log10(số ảnh + 1) theo nhãn.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
display(bar_plot([string(k) for k in 0:9], log10.(Float64.(class_counts .+ 1)), "Phân bố số ảnh theo lớp", joinpath(OUT, "01_class_distribution.svg"); ylabel="log10(số ảnh + 1)"))


### D05 — Tổng quan kiểm tra

- **Câu hỏi:** Miền cường độ và số ngoại lệ là bao nhiêu?
- **Cách tính:** In shape, miền pixel, ảnh đen/trắng và nhóm trùng.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
println("Dataset ",size(X),"; intensity ",Int(lo),":",Int(hi),"; counts ",class_counts)
println("Whole-black/white: ",length(black_ids),"/",length(white_ids),"; pixel-identical duplicate groups: ",length(dups))


### D08 — Chọn ảnh đại diện

- **Câu hỏi:** Có thể chọn lại đúng một ảnh mỗi lớp?
- **Cách tính:** Dùng seed 42, lưu image_id và metadata nguồn.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
rng=MersenneTwister(SEED);sample_ids=Int[]
for label in 0:9 c=findall(==(label),y);shuffle!(rng,c);push!(sample_ids,c[1]) end
write_csv(joinpath(OUT,"01_gallery_sample_ids.csv"),["label","class_name","image_id","source_split","source_index"],
    [(y[id],CLASSES[y[id]+1],id,split_name[id],source_idx[id]) for id in sample_ids])


### D09 — Gallery ảnh đại diện

In [ ]:
function sample_gallery(ids,path)
    io=IOBuffer()
    for (i,id) in enumerate(ids)
    x=12+mod(i-1,5)*205;yy=45+div(i-1,5)*190
    svg_text(io,x,yy,"$(y[id]) $(CLASSES[y[id]+1])";weight="bold")
    draw_image(io,@view(X[:,:,id]),x,yy+8)
    svg_text(io,x,yy+156,"ID $id | $(split_name[id]) $(source_idx[id])";size=10)
    end
    svg_save(path,"Seed 42 — one sample per class",String(take!(io)),1050,430)
end
display(sample_gallery(sample_ids,joinpath(OUT,"01_representative_gallery.svg")))


## 2. A — Cường độ và hai ngưỡng

Histogram toàn tập dùng 54.880.000 pixel. Q10+ lấy nearest-rank trên pixel dương; Otsu gồm pixel 0 và hòa điểm chọn ngưỡng thấp nhất.

### A01 — Histogram cường độ

- **Câu hỏi:** Mỗi mức xám xuất hiện bao nhiêu lần?
- **Cách tính:** Đếm toàn tập/theo lớp và kiểm tra tổng pixel.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
gh=zeros(Int64,256);ch=zeros(Int64,256,10)
for id in 1:N
    cl=y[id]+1
    for p in @view(X[:, :, id])
        b=Int(p)+1
        gh[b]+=1
        ch[b,cl]+=1
    end
end
@assert sum(gh)==N*784==54_880_000&&all(sum(@view ch[:,c])==5_488_000 for c in 1:10)


### A04–A05 — Q10+ và Otsu

- **Câu hỏi:** Hai phương pháp chọn ngưỡng ở giá trị nào?
- **Cách tính:** Tính phân vị thực nghiệm Q10+ và ngưỡng Otsu; tạo CDF.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
total=sum(gh);positive=total-gh[1];qrank=max(1,ceil(Int,.1*positive))
q10=hist_quantile(@view(gh[2:end]),qrank)+1
sw=sum((b-1)*gh[b] for b in 1:256);w0=0;s0=0.0;best=-Inf;otie=Int[]
for t in 0:254
    w0+=gh[t+1];s0+=t*gh[t+1];w1=total-w0
    if w0>0&&w1>0
    score=w0*w1*(s0/w0-(sw-s0)/w1)^2
    if score>best best=score;empty!(otie);push!(otie,t)
    elseif score==best push!(otie,t) end
    end
end
otsu=minimum(otie);
taus=(q10, otsu);
methods=("Q10+","Global Otsu");cdf=cumsum(gh)./total
@assert all(diff(cdf).>=0)&&isapprox(last(cdf),1.0)
zero_share=gh[1]/total;mean_intensity=sw/total;median_intensity=hist_quantile(gh,cld(total,2))


### A06 — Metadata ngưỡng

- **Câu hỏi:** Quy tắc tính có được lưu đủ để tái lập?
- **Cách tính:** Xuất threshold, CDF tại ngưỡng và quy tắc xử lý hòa điểm.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
threshold_rows=[]
for (name,tau) in zip(methods,taus)
    below=sum(@view gh[1:tau+1])
    rule=name=="Q10+" ? "positive pixels; nearest rank 10%; no interpolation" : "all pixels incl. 0; Otsu between-class variance; lowest tie"
    push!(threshold_rows,(name,tau,rule,below,total,below/total))
end
write_csv(joinpath(OUT,"02_thresholds.csv"),["method","threshold","rule","pixels_le_tau","total_pixels","cdf_at_tau"],threshold_rows)
open(joinpath(OUT,"02_threshold_metadata.toml"),"w") do io
    TOML.print(io,Dict("pixel_count"=>total,"zero_pixel_fraction"=>zero_share,"mean_intensity"=>mean_intensity,
    "median_intensity"=>median_intensity,"positive_pixel_count"=>positive,"q10_rank"=>qrank,"q10_positive"=>q10,
    "q10_rule"=>"nearest-rank empirical quantile on positive pixels; no interpolation","global_otsu"=>otsu,
    "otsu_tie_count"=>length(otie),"otsu_tie_rule"=>"lowest exact-score maximizing threshold","mask_rule"=>"I > tau");sorted=true)
end
println("Pixels $total; zero fraction $(round(100zero_share,digits=3))%; mean $(round(mean_intensity,digits=3)); median $median_intensity")
println("Q10+=$q10 (rank $qrank/$positive); Global Otsu=$otsu (ties $(length(otie))).")


### A03 — Histogram toàn tập

- **Câu hỏi:** Phân bố số pixel theo cường độ ra sao?
- **Cách tính:** Vẽ histogram log số đếm toàn tập.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
display(line_plot([(label="Số pixel (log10)",values=log10.(Float64.(gh) .+ 1),color="#3478b8")],
    "A1 Phân bố cường độ pixel toàn tập (log số đếm)",joinpath(OUT,"02_A1_global_histogram.svg");ylabel="log10(số pixel + 1)"))


### A04–A05 — Phân phối tích lũy toàn tập và theo lớp cùng ngưỡng tau

- **Câu hỏi:** Phân phối tích lũy toàn tập và phân phối tích lũy từng lớp khác nhau ra sao, và tau nằm ở đâu?
- **Cách tính:** Vẽ phân phối tích lũy toàn tập cùng 10 phân phối tích lũy theo lớp; đánh dấu tau_Q10+ và tau_Otsu bằng đường dọc nét đứt.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
palette=["#1768ac","#e4572e","#3a923a","#8e5ea2","#d89c18","#218c8c","#b83b5e","#536878","#a05a2c","#52796f"]
class_cdf=[]
for k in 0:9 push!(class_cdf,(label="$(k) $(CLASSES[k+1])",values=cumsum(@view(ch[:,k+1]))./sum(@view(ch[:,k+1])),color=palette[k+1])) end
cdf_series = vcat([(label="Toàn tập", values=cdf, color="#111827")], class_cdf)
display(line_plot_with_tau_markers(
    cdf_series,
    "A4–A5 Phân phối tích lũy cường độ toàn tập và theo lớp cùng hai ngưỡng tau",
    joinpath(OUT, "02_A4_A5_cdf_with_thresholds.svg");
    ylabel="Tỷ lệ tích lũy (0–1)",
    xmin=0,
    xmax=255,
    ymax=1,
    markers=[
        (Float64(q10), "tau Q10+ = $q10", "#d97706"),
        (Float64(otsu), "tau Otsu = $otsu", "#147d64"),
    ],
))


### A11–A13 — So sánh ngưỡng theo lớp

- **Câu hỏi:** Tỷ lệ pixel dưới ngưỡng khác nhau theo lớp?
- **Cách tính:** Tính tỷ lệ pixel <= tau, xuất bảng và so sánh Q10+/Otsu trên cùng biểu đồ; m=1 được vẽ phía trước m=2.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
class_tau=[]
for k in 0:9,m in 1:2
    h=@view ch[:,k+1];n=sum(h);below=sum(@view h[1:taus[m]+1])
    push!(class_tau,(k,CLASSES[k+1],methods[m],taus[m],below,n,below/n))
end
write_csv(joinpath(OUT,"02_A6_class_threshold_comparison.csv"),
    ["label","class_name","method","threshold","pixels_le_tau","class_pixels","fraction_le_tau"],class_tau)
class_tau_values = [
    [only(filter(row -> row[1] == k && row[3] == methods[m], class_tau))[7] for k in 0:9]
    for m in 1:2
]
display(overlaid_bar_plot(
    [string(k) for k in 0:9],
    class_tau_values,
    ["Q10+", "Otsu toàn cục "],
    "A6 Tỷ lệ pixel theo lớp không vượt ngưỡng",
    joinpath(OUT, "02_A6_class_threshold_comparison_overlay.svg");
    ylabel="Tỷ lệ pixel (0–1)",
    xlabel="Nhãn lớp",
    colors=["#d97706", "#147d64"],
))
write_csv(joinpath(OUT,"02_A7_decision_log.csv"),
    ["section","observation","evidence","interpretation","candidate_decision","validation_needed"],[
    ("A","One global threshold per method was calculated on combined data.","02_thresholds.csv; zero fraction $(zero_share); Q10+=$q10; Otsu=$otsu.","Intensity masks are foreground proxies, not semantic segmentation.","Carry both methods to B/C; record if numeric thresholds coincide.","Inspect masks and patch/gradient preservation before selection.")])


## 3. B — Tỷ lệ tiền cảnh

FR = số pixel có cường độ > tau chia 784; đây là proxy tiền cảnh, không phải phân đoạn ngữ nghĩa.

### B01–B02 — Foreground count và FR

- **Câu hỏi:** FR theo từng ảnh có nhất quán với CDF?
- **Cách tính:** Đếm foreground, chia 784 và kiểm tra miền [0,1].
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
fg=zeros(UInt16,N,2)
for id in 1:N,p in @view X[:,:,id]
    p>taus[1]&&(fg[id,1]+=1);p>taus[2]&&(fg[id,2]+=1)
end
fr=Float64.(fg)./784
@assert all((fr.>=0).&(fr.<=1))
for m in 1:2 @assert isapprox(sum(@view(fr[:,m]))/N,1-cdf[taus[m]+1];atol=1e-12) end
write_csv(joinpath(OUT,"03_B2_foreground_ratio_by_image.csv"),
    ["image_id","label","class_name","source_split","source_index","fr_q10","fr_otsu","delta_q10_minus_otsu"],
    [(id,y[id],CLASSES[y[id]+1],split_name[id],source_idx[id],fr[id,1],fr[id,2],fr[id,1]-fr[id,2]) for id in 1:N])


### B03 — Tóm tắt FR

- **Câu hỏi:** FR toàn tập và theo lớp khác nhau thế nào?
- **Cách tính:** Lưu mean, median, p05/p95 và min/max.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
frsum=[]
for m in 1:2 push!(frsum,(methods[m],summary5(@view fr[:,m])...)) end
write_csv(joinpath(OUT,"03_B2_foreground_ratio_summary.csv"),
    ["method","n","mean","median","p05","p95","minimum","maximum"],frsum)
class_fr=[]
for k in 0:9,m in 1:2
    ids=findall(==(k),y);push!(class_fr,(k,CLASSES[k+1],methods[m],summary5(@view fr[ids,m])...))
end
write_csv(joinpath(OUT,"03_B3_class_foreground_ratio.csv"),
    ["label","class_name","method","n","mean","median","p05","p95","minimum","maximum"],class_fr)


### B08 — Chênh lệch FR

- **Câu hỏi:** Hai ngưỡng cho kết quả tương tự đến đâu?
- **Cách tính:** Tính FR_Q10+ trừ FR_Otsu toàn tập/theo lớp.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
delta=fr[:,1].-fr[:,2];ord=sortperm(abs.(delta);rev=true)
write_csv(joinpath(OUT,"03_B4_delta_fr.csv"),["metric","value"],[
    ("definition","FR_Q10+ minus FR_Global_Otsu"),("images",N),("mean",sum(delta)/N),
    ("median",qlinear(delta,.5)),("p05",qlinear(delta,.05)),("p95",qlinear(delta,.95)),("max_abs",maximum(abs.(delta)))])


### B09 — Ảnh có delta lớn

- **Câu hỏi:** Ảnh nào nhạy nhất với ngưỡng?
- **Cách tính:** Lưu các image_id có trị tuyệt đối delta cao nhất.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
write_csv(joinpath(OUT,"03_B4_largest_delta_images.csv"),
    ["image_id","label","class_name","source_split","source_index","fr_q10","fr_otsu","delta_fr"],
    [(id,y[id],CLASSES[y[id]+1],split_name[id],source_idx[id],fr[id,1],fr[id,2],delta[id]) for id in ord[1:20]])


### B06–B07 — FR theo lớp

- **Câu hỏi:** Lớp nào có FR trung bình cao/thấp?
- **Cách tính:** Vẽ trung bình FR theo lớp cho hai phương pháp.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
series=[]
for m in 1:2
    vals=[only(filter(r->r[1]==k&&r[3]==methods[m],class_fr))[5] for k in 0:9]
    push!(series,(label=methods[m],values=vals,color=palette[m]))
end
display(line_plot(series,"B3 Tỷ lệ tiền cảnh trung bình theo lớp",joinpath(OUT,"03_B3_class_fr.svg");ylabel="FR trung bình (0–1)",xmin=0,xmax=9,ymax=1))


### B04–B05/B10 — Histogram và gallery mask

- **Câu hỏi:** Khác biệt FR có quan sát được trên mask?
- **Cách tính:** Vẽ histogram FR và đối chiếu ảnh gốc với hai mask.
- **Đầu vào/kết quả:** đọc các biến và output ở cell này.
- **Insight của tôi:** _Ghi nhận xét kèm số liệu hoặc image_id._
- **Cần kiểm chứng tiếp:** _Nêu giả thuyết và giới hạn bằng chứng._

In [ ]:
gallery_ids=unique(vcat(sample_ids,ord[1:5]));gallery_ids=gallery_ids[1:min(length(gallery_ids),15)]
function mask_gallery(ids,path)
    io=IOBuffer();rowh=174;w=470
    for (i,id) in enumerate(ids)
    yy=42+(i-1)*rowh;svg_text(io,10,yy,"ID $id | $(y[id]) $(CLASSES[y[id]+1]) | FR $(round(fr[id,1],digits=3)) / $(round(fr[id,2],digits=3))";size=11,weight="bold")
    draw_image(io,@view(X[:,:,id]),10,yy+8)
    for m in 1:2
    mask=map(v->v>taus[m],@view X[:,:,id]);x=165+(m-1)*145
    draw_image(io,mask,x,yy+8;binary=true);svg_text(io,x+70,yy+157,"$(methods[m]) tau=$(taus[m])";size=10,anchor="middle")
    end
    end
    svg_save(path,"B5 Ảnh gốc và mặt nạ theo hai ngưỡng",String(take!(io)),w,42+length(ids)*rowh)
end
display(mask_gallery(gallery_ids,joinpath(OUT,"03_B5_mask_gallery.svg")))
write_csv(joinpath(OUT,"03_B6_decision_log.csv"),
    ["section","observation","evidence","interpretation","candidate_decision","validation_needed"],[
    ("B","Per-image FR and sensitivity were calculated for both global masks.","03_B2 summary; mean Q10+=$(frsum[1][3]); mean Otsu=$(frsum[2][3]); delta sign is Q10+ minus Otsu.","A low FR does not prove the removed pixels are background.","Carry both methods into patch analysis; use original grayscale for descriptors.","Inspect high absolute delta images and preservation measures in C.")])


## Dữ liệu dùng chung — cùng nguồn cho hai kích thước

Ảnh gốc, hai ngưỡng intensity và FR ảnh ở trên được giữ chung. Các bảng lớn nằm ngoài Git. Không đọc output filtering cũ vì chúng dùng bộ α khác. Tổng gradient được định nghĩa là **tổng magnitude Sobel**, không phải tổng bình phương.

Mỗi hàm dưới đây phục vụ một nhóm nhiệm vụ; cell định nghĩa hàm chưa quét dataset.

### Hàm hỗ trợ — Bảng và biểu đồ dùng chung

In [ ]:
"""Ghi chữ SVG tự xuống dòng theo số ký tự; trả về số dòng để bố trí legend không đè nhau."""
function svg_wrapped_text(io, x, y, text; width=42, size=9, color="#172338")
    lines = String[]
    current = ""
    for word in split(text)
        if !isempty(current) && length(current) + length(word) + 1 > width
            push!(lines, current)
            current = ""
        end
        current = isempty(current) ? String(word) : current * " " * word
    end
    isempty(current) || push!(lines, current)
    for (index, line) in enumerate(lines)
        svg_text(io, x, y + (index - 1) * (size + 3), line; size=size, color=color)
    end
    return length(lines)
end

"""Lưu bảng NamedTuple vào CSV, hiển thị tối đa limit dòng và trả về đường dẫn."""
function evidence_table(name, rows; limit=24)
    isempty(rows) && error("Bảng $name không có dòng; cần công bố nhóm rỗng riêng.")
    header = string.(keys(first(rows)))
    path = joinpath(OUT, name * ".csv")
    write_csv(path, header, (Tuple(row) for row in rows))
    table = "| " * join(header, " | ") * " |\n"
    table *= "| " * join(fill("---", length(header)), " | ") * " |\n"
    for row in Iterators.take(rows, limit)
        cells = [replace(string(value), "|" => "\\|") for value in values(row)]
        table *= "| " * join(cells, " | ") * " |\n"
    end
    display(Markdown.parse(table))
    println("CSV: ", relpath(path, repo_root), " | ", length(rows), " dòng")
    return path
end

"""Tóm tắt vector số với quantile nội suy; đầu vào không được rỗng; trả về NamedTuple."""
function distribution_summary(values)
    data = sort!(Float64.(vec(collect(values))))
    isempty(data) && error("Không thể tóm tắt mẫu rỗng.")
    function percentile(probability)
        position = 1 + (length(data) - 1) * probability
        low = floor(Int, position)
        high = ceil(Int, position)
        return data[low] + (position - low) * (data[high] - data[low])
    end
    return (n=length(data), mean=sum(data) / length(data), minimum=first(data),
        p05=percentile(0.05), p25=percentile(0.25), median=percentile(0.5),
        p75=percentile(0.75), p95=percentile(0.95), maximum=last(data))
end

"""Đếm histogram cùng biên cho mọi chuỗi; trả về tâm bin và số đếm, giữ cả bin 0."""
function histogram_counts(values, edges)
    counts = zeros(Int, length(edges) - 1)
    for value in values
        bin = clamp(searchsortedlast(edges, value), 1, length(counts))
        counts[bin] += 1
    end
    @assert sum(counts) == length(values)
    return (x=(edges[1:end-1] .+ edges[2:end]) ./ 2, counts=counts)
end

"""Vẽ chuỗi có x/y riêng; trả về SVG, lưu theo name; mode=:scatter chỉ vẽ điểm."""
function evidence_plot(series, name, title; xlabel, ylabel, mode=:line)
    left, top, plot_width, plot_height = 85, 60, 650, 320
    xs = [Float64(x) for item in series for x in item.x if isfinite(x)]
    ys = [Float64(v) for item in series for v in item.y if isfinite(v)]
    isempty(xs) && error("Không có dữ liệu X cho $name.")
    isempty(ys) && error("Không có dữ liệu Y cho $name.")
    xmin, xmax = extrema(xs)
    ymin, ymax = min(0.0, minimum(ys)), maximum(ys)
    xmax = xmax > xmin ? xmax : xmin + 1
    ymax = ymax > ymin ? ymax : ymin + 1
    xpos(x) = left + plot_width * (x - xmin) / (xmax - xmin)
    ypos(v) = top + plot_height * (1 - (v - ymin) / (ymax - ymin))
    io = IOBuffer()
    for tick in 0:4
        x = xmin + (xmax - xmin) * tick / 4
        value = ymin + (ymax - ymin) * tick / 4
        svg_text(io, xpos(x), top + plot_height + 20, @sprintf("%.3g", x); anchor="middle", size=10)
        svg_text(io, left - 8, ypos(value) + 4, @sprintf("%.3g", value); anchor="end", size=10)
        println(io, "<line x1=\"$left\" y1=\"$(ypos(value))\" x2=\"$(left + plot_width)\" y2=\"$(ypos(value))\" stroke=\"#e2e8f0\"/>")
    end
    legend_y = 77
    for (index, item) in enumerate(series)
        points = String[]
        for (x, value) in zip(item.x, item.y)
            isfinite(x) && isfinite(value) || continue
            push!(points, "$(xpos(x)),$(ypos(value))")
            if mode == :scatter || length(item.x) == 1
                println(io, "<circle cx=\"$(xpos(x))\" cy=\"$(ypos(value))\" r=\"3\" fill=\"$(item.color)\" fill-opacity=\"0.65\"/>")
            end
        end
        if mode != :scatter
            println(io, "<polyline points=\"$(join(points, " "))\" fill=\"none\" stroke=\"$(item.color)\" stroke-width=\"2\"/>")
        end
        line_count = svg_wrapped_text(io, 755, legend_y, item.label; width=62, size=9, color=item.color)
        legend_y += max(line_count, 1) * 12 + 5
    end
    svg_text(io, left + plot_width / 2, 430, xlabel; anchor="middle")
    println(io, "<g transform=\"translate(18,220) rotate(-90)\">")
    svg_text(io, 0, 0, ylabel; anchor="middle")
    println(io, "</g>")
    return svg_save(joinpath(OUT, name * ".svg"), title, String(take!(io)), 1100, 455)
end

"""Vẽ histogram chung biên; values là các vector, labels là chú giải; trục Y log10(1+n)."""
function distribution_plot(values, labels, name, title; xlabel, bins=48)
    low = minimum(minimum, values)
    high = maximum(maximum, values)
    edges = collect(range(low, max(high, low + 1); length=bins + 1))
    series = []
    for (index, data) in enumerate(values)
        histogram = histogram_counts(data, edges)
        push!(series, (x=histogram.x, y=log10.(1 .+ histogram.counts),
            label=labels[index], color=palette[mod1(index, length(palette))]))
    end
    return evidence_plot(series, name, title; xlabel, ylabel="log10(1 + số đếm)")
end

"""Vẽ các ma trận cùng thang màu; entries chứa label/matrix; trả về SVG và lưu name."""
function evidence_maps(entries, name, title)
    columns = min(4, length(entries))
    rows = cld(length(entries), columns)
    finite_values = [Float64(v) for entry in entries for v in entry.matrix if !ismissing(v) && isfinite(v)]
    low = isempty(finite_values) ? 0.0 : minimum(finite_values)
    high = isempty(finite_values) ? 1.0 : maximum(finite_values)
    io = IOBuffer()
    for (index, entry) in enumerate(entries)
        x0 = 16 + mod(index - 1, columns) * 240
        y0 = 64 + div(index - 1, columns) * 220
        svg_wrapped_text(io, x0, y0, entry.label; width=38, size=9)
        matrix = entry.matrix
        cell = 140 / size(matrix, 1)
        for row in axes(matrix, 1), column in axes(matrix, 2)
            value = matrix[row, column]
            intensity = ismissing(value) ? 0.0 : (value - low) / max(high - low, eps())
            color = ismissing(value) ? "#cbd5e1" : "rgb($(round(Int, 250 - 210 * intensity)),$(round(Int, 250 - 140 * intensity)),240)"
            x = x0 + (column - 1) * cell
            yy = y0 + 36 + (row - 1) * cell
            println(io, "<rect x=\"$x\" y=\"$yy\" width=\"$cell\" height=\"$cell\" fill=\"$color\"/>")
        end
    end
    svg_text(io, 16, 47, @sprintf("Thang màu chung: sáng %.4g → xanh %.4g", low, high); size=10)
    return svg_save(joinpath(OUT, name * ".svg"), title, String(take!(io)), columns * 240, 65 + rows * 220)
end

### Hàm hỗ trợ — Dữ liệu patch và Sobel

In [ ]:
"""Tính Gx/Gy/magnitude Sobel ảnh grayscale; biên replicate; trả về ba ma trận Float64."""
function sobel_fields(image)
    height, width = size(image)
    gx = zeros(Float64, height, width)
    gy = similar(gx)
    for row in 1:height, column in 1:width
        r0, r2 = max(row - 1, 1), min(row + 1, height)
        c0, c2 = max(column - 1, 1), min(column + 1, width)
        gx[row, column] = Int(image[r0, c2]) + 2 * Int(image[row, c2]) + Int(image[r2, c2]) -
            Int(image[r0, c0]) - 2 * Int(image[row, c0]) - Int(image[r2, c0])
        gy[row, column] = Int(image[r2, c0]) + 2 * Int(image[r2, column]) + Int(image[r2, c2]) -
            Int(image[r0, c0]) - 2 * Int(image[r0, column]) - Int(image[r0, c2])
    end
    return (gx=gx, gy=gy, magnitude=hypot.(gx, gy))
end

"""Tạo bounds theo hàng trước/cột sau cho ảnh chia hết patch_size; trả về vector NamedTuple."""
function patch_bounds(height, width, patch_size)
    @assert height % patch_size == 0 && width % patch_size == 0
    columns = width ÷ patch_size
    return [(patch_id=(row - 1) * columns + column, patch_row=row, patch_col=column,
        r1=(row - 1) * patch_size + 1, r2=row * patch_size,
        c1=(column - 1) * patch_size + 1, c2=column * patch_size)
        for row in 1:(height ÷ patch_size) for column in 1:columns]
end

"""Điền metrics một ảnh vào arrays của một size; chuyên biệt kiểu ở vòng lặp pixel; trả về nothing."""
function measure_patch_image!(data, image, magnitude, id, thresholds)
    for bound in data.bounds
        total_value = 0.0
        square_value = 0.0
        gradient_value = 0.0
        count_q10 = 0
        count_otsu = 0
        for column in bound.c1:bound.c2, row in bound.r1:bound.r2
            value = Int(image[row, column])
            total_value += value
            square_value += value^2
            gradient_value += magnitude[row, column]
            count_q10 += value > thresholds[1]
            count_otsu += value > thresholds[2]
        end
        pid = bound.patch_id
        average = total_value / data.area
        data.mean[pid, id] = average
        data.variance[pid, id] = max(square_value / data.area - average^2, 0)
        data.gradient[pid, id] = gradient_value
        data.foreground[pid, id, 1] = count_q10
        data.foreground[pid, id, 2] = count_otsu
    end
    return nothing
end

"""Quét ảnh một lượt, tính Sobel chung và metrics hai kích thước; trả về patches/spatial/image_gradient."""
function build_patch_evidence(images, labels, thresholds; sizes=(4, 7))
    height, width, count_images = size(images)
    patches = Dict()
    for size_patch in sizes
        bounds = patch_bounds(height, width, size_patch)
        shape = (length(bounds), count_images)
        patches[size_patch] = (size=size_patch, area=size_patch^2, bounds=bounds,
            mean=zeros(Float32, shape), variance=zeros(Float32, shape),
            gradient=zeros(Float64, shape), foreground=zeros(UInt8, shape..., 2))
    end
    intensity_sum = zeros(Float64, height, width, 10)
    intensity_square = similar(intensity_sum)
    fill!(intensity_square, 0)
    gradient_sum = zeros(Float64, height, width, 10)
    occupancy_sum = zeros(Int, height, width, 10, 2)
    image_gradient = zeros(Float64, count_images)
    regional_per_image = []
    region_definitions = [
        ("bands3", "top", collect(CartesianIndices((1:9, 1:width)))),
        ("bands3", "middle", collect(CartesianIndices((10:19, 1:width)))),
        ("bands3", "bottom", collect(CartesianIndices((20:height, 1:width)))),
        ("quadrants", "top-left", collect(CartesianIndices((1:14, 1:14)))),
        ("quadrants", "top-right", collect(CartesianIndices((1:14, 15:width)))),
        ("quadrants", "bottom-left", collect(CartesianIndices((15:height, 1:14)))),
        ("quadrants", "bottom-right", collect(CartesianIndices((15:height, 15:width)))),
    ]
    for id in 1:count_images
        image = @view images[:, :, id]
        fields = sobel_fields(image)
        label = labels[id] + 1
        image_gradient[id] = sum(fields.magnitude)
        for (scheme, region, indices) in region_definitions
            push!(regional_per_image, (image_id=id, label=labels[id], scheme=scheme, region=region,
                mean_intensity=sum(Float64, image[indices]) / length(indices),
                mean_gradient=sum(fields.magnitude[indices]) / length(indices),
                fr_q10=count(>(thresholds[1]), image[indices]) / length(indices),
                fr_otsu=count(>(thresholds[2]), image[indices]) / length(indices)))
        end
        for column in 1:width, row in 1:height
            value = Float64(image[row, column])
            intensity_sum[row, column, label] += value
            intensity_square[row, column, label] += value^2
            gradient_sum[row, column, label] += fields.magnitude[row, column]
            for method in 1:2
                occupancy_sum[row, column, label, method] += value > thresholds[method]
            end
        end
        for data in values(patches)
            measure_patch_image!(data, image, fields.magnitude, id, thresholds)
        end
        id % 10_000 == 0 && println("Metrics hai kích thước: $id / $count_images")
    end
    class_counts = [count(==(label), labels) for label in 0:9]
    @assert all(>(0), class_counts)
    class_mean = intensity_sum ./ reshape(class_counts, 1, 1, 10)
    class_variance = max.(intensity_square ./ reshape(class_counts, 1, 1, 10) .- class_mean.^2, 0)
    class_gradient = gradient_sum ./ reshape(class_counts, 1, 1, 10)
    class_occupancy = occupancy_sum ./ reshape(class_counts, 1, 1, 10, 1)
    global_mean = dropdims(sum(intensity_sum; dims=3); dims=3) ./ count_images
    global_variance = max.(dropdims(sum(intensity_square; dims=3); dims=3) ./ count_images .- global_mean.^2, 0)
    return (patches=patches, image_gradient=image_gradient, regional_per_image=regional_per_image, spatial=(
        class_mean=class_mean, class_variance=class_variance, class_gradient=class_gradient,
        class_occupancy=class_occupancy, global_mean=global_mean, global_variance=global_variance,
        global_gradient=dropdims(sum(gradient_sum; dims=3); dims=3) ./ count_images,
        global_occupancy=dropdims(sum(occupancy_sum; dims=3); dims=3) ./ count_images))
end

"""Kiểm tra bounds, reconstruction, tổng gradient/foreground; trả về true hoặc ném AssertionError."""
function validate_patch_evidence(evidence, images, thresholds)
    for data in values(evidence.patches)
        @assert length(data.bounds) * data.area == size(images, 1) * size(images, 2)
        coverage = zeros(Int, size(images, 1), size(images, 2))
        for bound in data.bounds
            coverage[bound.r1:bound.r2, bound.c1:bound.c2] .+= 1
        end
        @assert all(==(1), coverage)
        for id in unique([1, size(images, 3) ÷ 2, size(images, 3)])
            rebuilt = zeros(UInt8, size(images, 1), size(images, 2))
            for bound in data.bounds
                rebuilt[bound.r1:bound.r2, bound.c1:bound.c2] .=
                    @view images[bound.r1:bound.r2, bound.c1:bound.c2, id]
            end
            @assert rebuilt == images[:, :, id]
            for method in 1:2
                @assert sum(data.foreground[:, id, method]) == count(>(thresholds[method]), images[:, :, id])
            end
        end
        @assert all(isapprox.(vec(sum(data.gradient; dims=1)), evidence.image_gradient; rtol=1e-12))
        @assert all(data.foreground .<= data.area)
    end
    return true
end

"""Ghi tuần tự metrics từng patch vào gzip, không tạo hàng triệu tuple RAM; trả về số dòng."""
function export_patch_metrics(path, patches, labels)
    row_count = 0
    temporary = path * ".partial"
    open(temporary, "w") do raw
        stream = GzipCompressorStream(raw)
        try
            println(stream, "image_id,class_label,patch_size,stride,patch_id,patch_row,patch_col,mean_intensity,intensity_variance,gradient_energy,mean_gradient_magnitude,foreground_count_q10,foreground_count_otsu,foreground_ratio_q10,foreground_ratio_otsu")
            for size_patch in sort(collect(keys(patches)))
                data = patches[size_patch]
                for id in eachindex(labels), bound in data.bounds
                    pid = bound.patch_id
                    first_count = data.foreground[pid, id, 1]
                    second_count = data.foreground[pid, id, 2]
                    row = (id, labels[id], size_patch, size_patch, pid, bound.patch_row, bound.patch_col,
                        data.mean[pid, id], data.variance[pid, id], data.gradient[pid, id],
                        data.gradient[pid, id] / data.area, first_count, second_count,
                        first_count / data.area, second_count / data.area)
                    println(stream, join(row, ','))
                    row_count += 1
                end
            end
        finally
            close(stream)
        end
    end
    mv(temporary, path; force=true)
    return row_count
end

### BASE01 — Tính metrics của hai kích thước và kiểm tra phép chia

In [ ]:
evidence = build_patch_evidence(X, y, taus; sizes=PATCH_SIZES)
patches = evidence.patches
spatial = evidence.spatial
image_gradient = evidence.image_gradient

@assert validate_patch_evidence(evidence, X, taus)
@assert size(patches[4].mean) == (49, N)
@assert size(patches[7].mean) == (16, N)
@assert sum(length(data.mean) for data in values(patches)) == 4_550_000

println("Đủ 4.550.000 patch; reconstruction, foreground và gradient đã đối chiếu.")

### BASE02 — Xuất bảng patch lớn và provenance

In [ ]:
metrics_path = joinpath(CACHE, "patch_metrics.csv.gz")
metrics_rows = export_patch_metrics(metrics_path, patches, y)
@assert metrics_rows == 4_550_000

provenance = Dict(
    "schema_version" => 1, "input_index_sha256" => idxsha,
    "additional_fixed_threshold_tau" => 10,
    "images" => N, "patch_sizes" => collect(PATCH_SIZES), "strides" => collect(PATCH_SIZES),
    "rows" => metrics_rows, "alpha" => collect(ALPHAS), "thresholds" => collect(taus),
    "seed" => SEED, "metrics_sha256" => bytes2hex(open(sha256, metrics_path)),
    "gradient_definition" => "sum(hypot(Gx,Gy)); original grayscale; Sobel replicate",
    "julia_version" => string(VERSION), "opencv_julia_version" => string(Base.pkgversion(OpenCV)),
    "opencv_version" => OpenCV.getVersionString(), "metrics_path" => relpath(metrics_path, repo_root),
)
open(joinpath(CACHE, "patch_metrics_metadata.toml"), "w") do io
    TOML.print(io, provenance; sorted=true)
end
println(relpath(metrics_path, repo_root), " | ", metrics_rows, " dòng")

### BASE03 — Pilot cân bằng và ID đại diện

In [ ]:
pilot_rng = MersenneTwister(SEED)
pilot = Int[]
for label in 0:9
    candidates = findall(==(label), y)
    shuffle!(pilot_rng, candidates)
    append!(pilot, candidates[1:PILOT_PER_CLASS])
end
@assert length(pilot) == 200
@assert all(count(id -> y[id] == label, pilot) == 20 for label in 0:9)

pilot_manifest = [(image_id=id, label=y[id], source_split=split_name[id],
    source_index=source_idx[id], seed=SEED) for id in pilot]
evidence_table("pilot_manifest", pilot_manifest; limit=10)

### Hàm hỗ trợ — Gallery hai kích thước và ví dụ patch

In [ ]:
"""Vẽ cùng ảnh với hai cách chia patch; có thể tô viền patch bị loại theo method/alpha; trả về SVG."""
function comparison_gallery(ids, images, labels, patches, name; method=0, alpha=0.0)
    io = IOBuffer()
    scale = 3
    panel_width = 660
    panel_height = 265
    for (index, id) in enumerate(ids)
        y0 = 46 + (index - 1) * panel_height
        svg_text(io, 12, y0, "Lớp $(labels[id]) — $(CLASSES[labels[id] + 1]) | ID $id"; weight="bold")
        draw_image(io, @view(images[:, :, id]), 12, y0 + 20; scale=scale)
        svg_text(io, 12, y0 + 120, "Ảnh gốc"; size=10)
        for (size_index, size_patch) in enumerate(sort(collect(keys(patches))))
            data = patches[size_patch]
            x0 = 125 + (size_index - 1) * 265
            tile_width = size_patch * scale + 8
            tile_height = size_patch * scale + 14
            svg_text(io, x0, y0 + 15, "Patch $(size_patch)×$(size_patch), $(length(data.bounds)) ô"; size=11)
            for bound in data.bounds
                x = x0 + (bound.patch_col - 1) * tile_width
                yy = y0 + 28 + (bound.patch_row - 1) * tile_height
                patch = @view images[bound.r1:bound.r2, bound.c1:bound.c2, id]
                draw_image(io, patch, x, yy; scale=scale)
                dropped = method > 0 && data.foreground[bound.patch_id, id, method] <= floor(Int, alpha * data.area)
                color = dropped ? "#e11d48" : "#147d64"
                side = size_patch * scale
                println(io, "<rect x=\"$x\" y=\"$yy\" width=\"$side\" height=\"$side\" fill=\"none\" stroke=\"$color\" stroke-width=\"1.5\"/>")
                svg_text(io, x, yy + side + 10, string(bound.patch_id); size=8)
            end
        end
    end
    title = method == 0 ? "Cùng ảnh, hai kích thước patch — thứ tự theo hàng" :
        "Patch giữ (xanh) / loại (đỏ): $(methods[method]), α=$alpha"
    return svg_save(joinpath(OUT, name * ".svg"), title, String(take!(io)), panel_width, 48 + length(ids) * panel_height)
end

"""Vẽ các patch được chọn cùng magnitude Sobel; examples chứa size/id/pid/group; trả về SVG."""
function patch_example_gallery(examples, images, patches, name)
    io = IOBuffer()
    for (index, example) in enumerate(examples)
        data = patches[example.size]
        bound = data.bounds[example.pid]
        image = @view images[:, :, example.id]
        patch = @view image[bound.r1:bound.r2, bound.c1:bound.c2]
        magnitude = sobel_fields(image).magnitude[bound.r1:bound.r2, bound.c1:bound.c2]
        # Chuẩn hóa hiển thị theo giới hạn Sobel chung, không theo max riêng của patch.
        gradient_image = round.(UInt8, clamp.(magnitude ./ (4 * 255 * sqrt(2)), 0, 1) .* 255)
        yy = 55 + (index - 1) * 95
        svg_text(io, 12, yy, "$(example.group) | $(example.size)×$(example.size) | ID $(example.id), patch $(example.pid)"; size=11)
        draw_image(io, patch, 16, yy + 9; scale=56 / data.size)
        draw_image(io, gradient_image, 90, yy + 9; scale=56 / data.size)
        svg_text(io, 165, yy + 28, "variance=$(round(data.variance[example.pid, example.id]; digits=2))"; size=10)
        svg_text(io, 165, yy + 45, "mean gradient=$(round(data.gradient[example.pid, example.id] / data.area; digits=2))"; size=10)
    end
    return svg_save(joinpath(OUT, name * ".svg"), "Patch gốc (trái), magnitude Sobel (phải)", String(take!(io)), 650, 60 + 95 * length(examples))
end

"""Chọn ví dụ theo quantile variance/gradient cùng kích thước; trả về ví dụ và bảng nhóm kể cả nhóm rỗng."""
function select_patch_groups(patches; limit=2)
    examples, rows = [], []
    for size_patch in sort(collect(keys(patches)))
        data = patches[size_patch]
        variance = distribution_summary(data.variance)
        gradient = distribution_summary(data.gradient ./ data.area)
        groups = [("Variance thấp, gradient thấp", (v, g) -> v <= variance.p25 && g <= gradient.p25),
            ("Variance cao, gradient cao", (v, g) -> v > variance.p75 && g > gradient.p75),
            ("Variance cao, gradient thấp", (v, g) -> v > variance.p75 && g <= gradient.p25)]
        for (name, predicate) in groups
            selected = 0
            total = 0
            for id in axes(data.mean, 2), pid in axes(data.mean, 1)
                if predicate(data.variance[pid, id], data.gradient[pid, id] / data.area)
                    total += 1
                    if selected < limit
                        push!(examples, (size=size_patch, id=id, pid=pid, group=name))
                        selected += 1
                    end
                end
            end
            push!(rows, (patch_size=size_patch, group=name, patches=total,
                variance_p25=variance.p25, variance_p75=variance.p75,
                mean_gradient_p25=gradient.p25, mean_gradient_p75=gradient.p75))
        end
    end
    return (examples=examples, groups=rows)
end

## I1 — Patch Suitability: patch giữ được cấu trúc cục bộ nào?

### I1.0 — Cùng 10 ảnh đại diện, hai cách cắt

In [ ]:
display(comparison_gallery(sample_ids, X, y, patches, "i1_representative_patch_comparison"))

**Cách đọc:** Mỗi hàng cùng một ảnh: ảnh gốc, 49 patch 4×4 và 16 patch 7×7; ID patch theo hàng trước/cột sau.

**Câu hỏi gợi ý:** Chi tiết nào nằm trong một patch? Chi tiết nào bị chia qua nhiều patch?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q1.1 — Patch biến thiên cường độ như thế nào?

**Quyết định TC:** Đánh giá raw descriptor và số patch đồng nhất trước K-means.

**Cách tính và đầu vào:** Mean/variance từ pixel gốc. Variance bằng 0 được tách tối (mean=0) và sáng hơn 0; gần 0: 0<variance≤1.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
variance_rows = [merge((patch_size=size_patch,), distribution_summary(patches[size_patch].variance))
    for size_patch in PATCH_SIZES]
evidence_table("q11_variance_summary", variance_rows)
display(distribution_plot([vec(patches[s].variance) for s in PATCH_SIZES],
    ["Patch $(s)×$(s)" for s in PATCH_SIZES], "q11_variance_distribution",
    "Q1.1 Phân phối variance của patch"; xlabel="Variance cường độ, đơn vị intensity²"))

In [ ]:
homogeneous_rows = []
for size_patch in PATCH_SIZES
    data = patches[size_patch]
    groups = (("Đồng nhất tối", (v, m) -> v == 0 && m == 0),
        ("Đồng nhất sáng hơn 0", (v, m) -> v == 0 && m > 0),
        ("Gần đồng nhất", (v, m) -> 0 < v <= 1), ("Có biến thiên", (v, m) -> v > 1))
    for (name, predicate) in groups
        mask = predicate.(data.variance, data.mean)
        total = count(mask)
        push!(homogeneous_rows, (patch_size=size_patch, group=name, count=total,
            fraction=total / length(mask), mean_intensity=total == 0 ? missing : sum(data.mean[mask]) / total,
            mean_fr_q10=total == 0 ? missing : sum(data.foreground[:, :, 1][mask]) / (total * data.area),
            mean_fr_otsu=total == 0 ? missing : sum(data.foreground[:, :, 2][mask]) / (total * data.area)))
    end
end
evidence_table("q11_homogeneous_groups", homogeneous_rows)

**Cách đọc:** X là variance cường độ, Y là log10(1+số patch); bảng dùng tỷ lệ trên toàn bộ patch cùng size.

**Câu hỏi gợi ý:** Variance=0 thuộc nhóm tối hay sáng? Có nên coi tất cả patch variance thấp là vô ích?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q1.2 — Patch chứa cấu trúc cạnh không?

**Quyết định TC:** Đánh giá thông tin cục bộ và bằng chứng cho gradient descriptor.

**Cách tính và đầu vào:** Sobel ảnh gốc dùng chung. So sánh mean magnitude/pixel; tổng gradient mỗi patch được lưu riêng vì diện tích khác nhau.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
gradient_rows = [merge((patch_size=s, quantity="mean_magnitude_per_pixel",),
    distribution_summary(patches[s].gradient ./ patches[s].area)) for s in PATCH_SIZES]
evidence_table("q12_mean_gradient_summary", gradient_rows)
energy_rows = [merge((patch_size=s, quantity="sum_magnitude_per_patch",),
    distribution_summary(patches[s].gradient)) for s in PATCH_SIZES]
evidence_table("q12_gradient_energy_summary", energy_rows)
display(distribution_plot([vec(patches[s].gradient ./ patches[s].area) for s in PATCH_SIZES],
    ["Patch $(s)×$(s)" for s in PATCH_SIZES], "q12_mean_gradient_distribution",
    "Q1.2 Mức biến thiên cạnh theo pixel trong patch"; xlabel="Mean magnitude Sobel / pixel"))

In [ ]:
gradient_examples = []
for s in PATCH_SIZES
    data = patches[s]
    statistics = distribution_summary(data.gradient ./ data.area)
    for (name, target) in (("Gradient p05", statistics.p05), ("Gradient p95", statistics.p95))
        location = argmin(abs.(data.gradient ./ data.area .- target))
        push!(gradient_examples, (size=s, id=location[2], pid=location[1], group=name))
    end
end
display(patch_example_gallery(gradient_examples, X, patches, "q12_gradient_examples"))

**Cách đọc:** Magnitude hiển thị theo cùng giới hạn Sobel; ví dụ gần p05/p95 toàn tập từng size, không chuẩn hóa sáng riêng từng patch.

**Câu hỏi gợi ý:** Patch gradient cao nằm trên đường viền hay texture? Size lớn có giữ cấu trúc trong cùng một patch rõ hơn không?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q1.3 — Thông tin tập trung ở đâu trên ảnh?

**Quyết định TC:** Bổ sung bằng chứng về kích thước patch và spatial encoding.

**Cách tính và đầu vào:** Trung bình variance và mean gradient từng vị trí patch, toàn tập và từng lớp; cùng thang màu trong một hình.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
position_rows = []
for s in PATCH_SIZES, label in -1:9
    data = patches[s]
    ids = label == -1 ? collect(1:N) : findall(==(label), y)
    for bound in data.bounds
        pid = bound.patch_id
        push!(position_rows, (patch_size=s, label=label, patch_id=pid,
            row=bound.patch_row, column=bound.patch_col,
            mean_variance=sum(data.variance[pid, ids]) / length(ids),
            mean_gradient=sum(data.gradient[pid, ids]) / (length(ids) * data.area)))
    end
end
evidence_table("q13_patch_position_metrics", position_rows; limit=8)

In [ ]:
for (metric, title) in ((:mean_variance, "Variance cường độ"), (:mean_gradient, "Mean gradient / pixel"))
    entries = []
    for label in -1:9, s in PATCH_SIZES
        rows = filter(row -> row.patch_size == s && row.label == label, position_rows)
        side = H ÷ s
        matrix = [only(filter(row -> row.row == r && row.column == c, rows))[metric]
            for r in 1:side, c in 1:side]
        push!(entries, (label="$(label == -1 ? "Toàn tập" : "Lớp $label") | patch $(s)×$(s)", matrix=matrix))
    end
    display(evidence_maps(entries, "q13_$(metric)_maps", "Q1.3 $title theo vị trí patch"))
end

**Cách đọc:** Các ô là vị trí patch, phía trên hình ứng với phía trên ảnh. Ma trận 7×7 và 4×4 thể hiện cùng phạm vi ảnh 28×28.

**Câu hỏi gợi ý:** Thông tin tập trung ở giữa hay rìa? Trung bình toàn tập có che khác biệt giữa lớp không?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Hàm hỗ trợ — Filtering, cutoff rời rạc và đường biên đánh đổi

In [ ]:
"""Tính giữ/loại cho một cấu hình; data là metrics một kích thước; trả về vector cấp ảnh."""
function filter_outcome(data, method, alpha)
    count_images = size(data.mean, 2)
    retained = zeros(Int, count_images)
    lost_gradient = zeros(Float64, count_images)
    lost_foreground = zeros(Int, count_images)
    cutoff = method == 0 ? -1 : floor(Int, alpha * data.area)
    for id in 1:count_images, pid in axes(data.mean, 1)
        if method == 0 || data.foreground[pid, id, method] > cutoff
            retained[id] += 1
        else
            lost_gradient[id] += data.gradient[pid, id]
            lost_foreground[id] += data.foreground[pid, id, method]
        end
    end
    return (retained=retained, lost_gradient=lost_gradient,
        lost_foreground=lost_foreground, cutoff=cutoff)
end

"""Tóm tắt filtering cho tập image IDs, có thể toàn tập hoặc theo lớp; trả về NamedTuple."""
function filter_summary(data, outcome, ids, image_gradient, method, alpha)
    original_patches = length(ids) * length(data.bounds)
    kept = sum(outcome.retained[ids])
    total_gradient = sum(image_gradient[ids])
    total_foreground = sum(data.foreground[:, ids, max(method, 1)])
    return (patch_size=data.size, method=method == 0 ? "No-filter" : methods[method],
        alpha=method == 0 ? missing : alpha, cutoff=outcome.cutoff, images=length(ids),
        retained=kept, reduction=1 - kept / original_patches,
        mean_retained=kept / length(ids), empty_rate=count(iszero, outcome.retained[ids]) / length(ids),
        retained_pixel_slots=kept * data.area, raw_descriptor_dimensions=kept * data.area,
        edge_loss=total_gradient == 0 ? missing : sum(outcome.lost_gradient[ids]) / total_gradient,
        foreground_loss=total_foreground == 0 ? missing : sum(outcome.lost_foreground[ids]) / total_foreground)
end

"""Đánh giá sáu cấu hình + baseline mỗi size; trả về bảng global/class/distribution và outcomes."""
function evaluate_filters(patches, labels, image_gradient, alphas)
    global_rows, class_rows, distributions = [], [], []
    outcomes = Dict()
    for size_patch in sort(collect(keys(patches)))
        data = patches[size_patch]
        settings = vcat([(method=0, alpha=0.0)],
            [(method=method, alpha=alpha) for method in 1:2 for alpha in alphas])
        for setting in settings
            outcome = filter_outcome(data, setting.method, setting.alpha)
            outcomes[(size_patch, setting.method, setting.alpha)] = outcome
            summary = filter_summary(data, outcome, eachindex(labels), image_gradient, setting.method, setting.alpha)
            push!(global_rows, summary)
            for label in 0:9
                ids = findall(==(label), labels)
                push!(class_rows, merge((label=label,),
                    filter_summary(data, outcome, ids, image_gradient, setting.method, setting.alpha)))
            end
            for retained in 0:length(data.bounds)
                push!(distributions, (patch_size=size_patch, method=summary.method, alpha=summary.alpha,
                    retained=retained, image_count=count(==(retained), outcome.retained)))
            end
        end
        for method in 1:2, index in 2:length(alphas)
            @assert all(outcomes[(size_patch, method, alphas[index])].retained .<=
                outcomes[(size_patch, method, alphas[index - 1])].retained)
        end
        baseline = outcomes[(size_patch, 0, 0.0)]
        @assert all(==(length(data.bounds)), baseline.retained)
        @assert all(iszero, baseline.lost_gradient) && all(iszero, baseline.lost_foreground)
    end
    return (global_rows=global_rows, class_rows=class_rows,
        distributions=distributions, outcomes=outcomes)
end

"""Quét mọi cutoff foreground bằng histogram, trả về reduction/loss và cờ Pareto."""
function filter_frontiers(patches, image_gradient)
    rows = []
    total_gradient = sum(image_gradient)
    for size_patch in sort(collect(keys(patches))), method in 1:2
        data = patches[size_patch]
        counts = zeros(Int, data.area + 1)
        gradient = zeros(Float64, data.area + 1)
        for id in axes(data.mean, 2), pid in axes(data.mean, 1)
            bin = Int(data.foreground[pid, id, method]) + 1
            counts[bin] += 1
            gradient[bin] += data.gradient[pid, id]
        end
        cumulative_count, cumulative_gradient = cumsum(counts), cumsum(gradient)
        for cutoff in -1:data.area
            dropped = cutoff == -1 ? 0 : cumulative_count[cutoff + 1]
            lost = cutoff == -1 ? 0.0 : cumulative_gradient[cutoff + 1]
            push!(rows, (patch_size=size_patch, method=methods[method], cutoff=cutoff,
                effective_alpha=cutoff / data.area, reduction=dropped / length(data.mean),
                edge_loss=total_gradient == 0 ? missing : lost / total_gradient))
        end
    end
    return [merge(row, (pareto=ismissing(row.edge_loss) ? false : !any(other ->
        !ismissing(other.edge_loss) && other.reduction >= row.reduction && other.edge_loss <= row.edge_loss &&
        (other.reduction > row.reduction || other.edge_loss < row.edge_loss), rows),)) for row in rows]
end

"""So sánh τ cố định với Q10+/Otsu cho hai size; trả về bảng global/class và cutoff counts."""
function compare_threshold_filtering(images, patches, labels, image_gradient, fixed_tau, alphas)
    methods_to_compare = [(name="τ=$fixed_tau", tau=fixed_tau, method_index=0),
        (name=methods[1], tau=taus[1], method_index=1),
        (name=methods[2], tau=taus[2], method_index=2)]
    global_rows, class_rows = [], []
    foreground_counts = Dict()
    for size_patch in sort(collect(keys(patches)))
        data = patches[size_patch]
        for setting in methods_to_compare
            counts = if setting.method_index == 1 || setting.method_index == 2
                data.foreground[:, :, setting.method_index]
            else
                fixed = zeros(UInt8, length(data.bounds), size(images, 3))
                for id in axes(images, 3), bound in data.bounds
                    fixed[bound.patch_id, id] = count(>(setting.tau),
                        @view images[bound.r1:bound.r2, bound.c1:bound.c2, id])
                end
                fixed
            end
            foreground_counts[(size_patch, setting.name)] = counts
            for alpha in alphas
                cutoff = floor(Int, alpha * data.area)
                retained = zeros(Int, length(labels))
                lost_gradient = zeros(Float64, length(labels))
                lost_foreground = zeros(Int, length(labels))
                for id in eachindex(labels), pid in axes(counts, 1)
                    if counts[pid, id] > cutoff
                        retained[id] += 1
                    else
                        lost_gradient[id] += data.gradient[pid, id]
                        lost_foreground[id] += counts[pid, id]
                    end
                end
                for label in -1:9
                    ids = label == -1 ? collect(eachindex(labels)) : findall(==(label), labels)
                    patch_total = length(ids) * length(data.bounds)
                    grad_total = sum(image_gradient[ids])
                    fg_total = sum(counts[:, ids])
                    row = (patch_size=size_patch, tau=setting.tau, method=setting.name,
                        alpha=alpha, cutoff_pixels=cutoff, label=label, images=length(ids),
                        retained=sum(retained[ids]), reduction=1 - sum(retained[ids]) / patch_total,
                        mean_retained=sum(retained[ids]) / length(ids),
                        empty_rate=count(iszero, retained[ids]) / length(ids),
                        edge_loss=grad_total == 0 ? missing : sum(lost_gradient[ids]) / grad_total,
                        foreground_loss=fg_total == 0 ? missing : sum(lost_foreground[ids]) / fg_total)
                    push!(label == -1 ? global_rows : class_rows, row)
                end
            end
        end
    end
    return (global_rows=global_rows, class_rows=class_rows, foreground_counts=foreground_counts)
end


## I2 — Background Filtering: giảm patch và mất thông tin

Giữ `FR>α`, tương đương loại `foreground_count≤floor(α×area)`. Dùng cùng α={0,0.0625,0.125}: patch 4×4 loại count 0/≤1/≤2; patch 7×7 loại count 0/≤3/≤6. No-filter giữ mọi patch, khác α=0.

Mask là foreground giả định; EdgeLoss là proxy cho mất cấu trúc cạnh, không phải ground truth hình dạng.

### Q2.1 — Bao nhiêu patch trống hoặc gần trống?

**Quyết định TC:** Xem có cơ sở thử lọc patch trước descriptor/K-means không.

**Cách tính và đầu vào:** FR riêng từng size và τ; near-empty dùng cùng tỷ lệ 6.25%/12.5%, công bố cutoff thực tế.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
empty_rows = []
fr_series = []
for (size_index, s) in enumerate(PATCH_SIZES), method in 1:2
    data = patches[s]
    counts = [count(==(value), data.foreground[:, :, method]) for value in 0:data.area]
    push!(fr_series, (x=collect(0:data.area) ./ data.area, y=log10.(1 .+ counts),
        label="$(s)×$(s), $(methods[method])", color=palette[2 * (size_index - 1) + method]))
    for alpha in ALPHAS
        cutoff = floor(Int, alpha * data.area)
        near_count = vec(sum(data.foreground[:, :, method] .<= cutoff; dims=1))
        push!(empty_rows, (patch_size=s, method=methods[method], alpha=alpha, cutoff=cutoff,
            patch_fraction=sum(near_count) / length(data.mean), images_with_near_empty=count(>(0), near_count) / N))
    end
end
evidence_table("q21_empty_near_empty", empty_rows)
display(evidence_plot(fr_series, "q21_patch_fr_distribution", "Q2.1 Phân phối foreground ratio của patch";
    xlabel="FR = số pixel foreground / diện tích patch", ylabel="log10(1 + số patch)"))

In [ ]:
near_distributions = []
for s in PATCH_SIZES, method in 1:2, alpha in ALPHAS
    data = patches[s]
    cutoff = floor(Int, alpha * data.area)
    counts_per_image = vec(sum(data.foreground[:, :, method] .<= cutoff; dims=1))
    for value in 0:length(data.bounds)
        push!(near_distributions, (patch_size=s, method=methods[method], alpha=alpha,
            near_empty_patches_per_image=value, image_count=count(==(value), counts_per_image)))
    end
end
evidence_table("q21_near_empty_per_image_distribution", near_distributions; limit=10)

**Cách đọc:** FR của patch 4×4 có bước 1/16; patch 7×7 có bước 1/49. Y của hình là log số đếm, không phải xác suất.

**Câu hỏi gợi ý:** Hai kích thước có bao nhiêu patch FR=0? Số patch gần trống có phân bố đồng đều giữa các ảnh không?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q2.2 — Filtering giảm bao nhiêu patch?

**Quyết định TC:** Chọn cấu hình thử nghiệm có khả năng giảm chi phí.

**Cách tính và đầu vào:** Đánh giá 14 phương án; báo patch reduction và retained pixel slots để không chỉ so số descriptor khác kích thước.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
filters = evaluate_filters(patches, y, image_gradient, ALPHAS)
evidence_table("q22_filter_comparison", filters.global_rows)
evidence_table("q22_retained_distribution", filters.distributions; limit=10)

distinct_rows = []
for s in PATCH_SIZES
    masks = [vec(patches[s].foreground[:, :, m] .> floor(Int, a * patches[s].area))
        for m in 1:2 for a in ALPHAS]
    push!(distinct_rows, (patch_size=s, nominal_filtered=6,
        distinct_keep_masks=length(unique(masks)), separate_no_filter=true))
end
evidence_table("q22_distinct_filter_masks", distinct_rows)

In [ ]:
efficiency_series = [(x=[index], y=[row.reduction],
    label="$(row.patch_size)×$(row.patch_size), $(row.method), α=$(row.alpha)",
    color=palette[mod1(index, length(palette))]) for (index, row) in enumerate(filters.global_rows)]
display(evidence_plot(efficiency_series, "q22_patch_reduction", "Q2.2 Mức giảm patch của 14 phương án";
    xlabel="STT phương án theo bảng q22_filter_comparison", ylabel="Patch reduction (0–1)", mode=:scatter))

**Cách đọc:** 14 phương án gồm hai baseline no-filter. Retained pixel slots cũng là số chiều raw descriptor trước khi lượng tử hóa.

**Câu hỏi gợi ý:** Giảm patch có đi cùng giảm số pixel descriptor tương ứng không? Có ảnh nào hết patch sau lọc?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q2.2b — Bổ sung ngưỡng cố định τ=10

**Câu hỏi:** τ=10 giảm được bao nhiêu patch và giữ cạnh ra sao so với Q10+ (29) và Otsu toàn cục (97)?

**Cách tính:** Tạo count patch `I>10` từ ảnh grayscale gốc; dùng cùng α={0; 0,0625; 0,125}, cùng patch sizes và cùng định nghĩa EdgeLoss. Bảng global dùng label=-1; bảng theo lớp giúp thấy tác động không đồng đều.

**Giới hạn:** τ=10 là ngưỡng thử do người dùng yêu cầu, không được chọn từ quantile/Otsu và chưa được tối ưu.

In [ ]:
tau10_results = compare_threshold_filtering(X, patches, y, image_gradient, 10, ALPHAS)
tau10_global = tau10_results.global_rows
tau10_class = tau10_results.class_rows

tau10_table = [(patch_size=row.patch_size, tau=row.tau, method=row.method,
    alpha=row.alpha, cutoff_pixels=row.cutoff_pixels, reduction=row.reduction,
    edge_loss=row.edge_loss, foreground_loss=row.foreground_loss,
    mean_retained=row.mean_retained, empty_rate=row.empty_rate)
    for row in tau10_global]
evidence_table("q22_tau10_vs_q10_otsu", tau10_table)
evidence_table("q22_tau10_class_impact", tau10_class; limit=18)

for size_patch in PATCH_SIZES
    series = []
    for (index, method_name) in enumerate(("τ=10", methods[1], methods[2]))
        rows = sort(filter(row -> row.patch_size == size_patch && row.method == method_name, tau10_global);
            by=row -> row.alpha)
        push!(series, (x=[row.reduction for row in rows], y=[row.edge_loss for row in rows],
            label=method_name, color=palette[index]))
    end
    display(evidence_plot(series, "q22_tau10_tradeoff_$(size_patch)x$(size_patch)",
        "Q2.2 τ=10 so với Q10+/Otsu — patch $(size_patch)×$(size_patch)";
        xlabel="Patch reduction (0–1)", ylabel="EdgeLoss (0–1)", mode=:scatter))
end

**Cách đọc:** Mỗi điểm là một α. Reduction càng lớn thì loại càng nhiều patch; EdgeLoss càng thấp thì giữ lại càng nhiều tổng gradient gốc.

**Câu hỏi gợi ý:** τ=10 có giảm được nhiều patch hơn Q10+? Khi reduction gần nhau, ngưỡng nào mất ít gradient hơn?

**Insight của tôi:** _Ghi nhận số liệu và chọn cấu hình cần đem sang TC để đối chứng._

**Cần kiểm chứng tiếp:** _τ=10 là ngưỡng thử; so sánh utility sau codebook/transaction trước khi chọn._

### Q2.3 — Filtering làm mất cạnh hoặc chi tiết nhỏ không?

**Quyết định TC:** So sánh mức bảo toàn thông tin với lợi ích giảm dữ liệu.

**Cách tính và đầu vào:** EdgeLoss = tổng magnitude trong patch loại / tổng magnitude ảnh gốc. So sánh cùng α và các cutoff nguyên đạt reduction gần nhau.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
loss_series = [(x=[row.reduction], y=[Float64(row.edge_loss)],
    label="$(row.patch_size)×$(row.patch_size), $(row.method), α=$(row.alpha)",
    color=palette[mod1(index, length(palette))]) for (index, row) in enumerate(filters.global_rows)
    if !ismissing(row.edge_loss)]
display(evidence_plot(loss_series, "q23_reduction_edge_loss", "Q2.3 Đánh đổi giảm patch và mất gradient";
    xlabel="Patch reduction (0–1)", ylabel="EdgeLoss (0–1)", mode=:scatter))

In [ ]:
frontiers = filter_frontiers(patches, image_gradient)
evidence_table("q23_all_integer_cutoffs", frontiers; limit=10)
frontier_series = []
for (index, (s, method)) in enumerate([(s, m) for s in PATCH_SIZES for m in methods])
    rows = filter(row -> row.patch_size == s && row.method == method && !ismissing(row.edge_loss), frontiers)
    push!(frontier_series, (x=[r.reduction for r in rows], y=[r.edge_loss for r in rows],
        label="$(s)×$(s), $method", color=palette[index]))
end
display(evidence_plot(frontier_series, "q23_cutoff_tradeoff", "Q2.3 Các điểm reduction–loss đạt được khi quét cutoff";
    xlabel="Patch reduction (0–1)", ylabel="EdgeLoss (0–1)", mode=:scatter))

In [ ]:
near_matches = []
for row in frontiers
    row.patch_size == 4 && !ismissing(row.edge_loss) || continue
    candidates = filter(other -> other.patch_size == 7 && other.method == row.method && !ismissing(other.edge_loss), frontiers)
    for candidate_index in sortperm(abs.([other.reduction - row.reduction for other in candidates]))[1:min(2, length(candidates))]
        other = candidates[candidate_index]
        push!(near_matches, (method=row.method, cutoff_4=row.cutoff, reduction_4=row.reduction,
            cutoff_7=other.cutoff, reduction_7=other.reduction, reduction_gap=other.reduction - row.reduction,
            edge_loss_4=row.edge_loss, edge_loss_7=other.edge_loss))
    end
end
evidence_table("q23_near_reduction_comparisons", near_matches; limit=10)

In [ ]:
for method in 1:2, alpha in ALPHAS
    extreme_ids = Int[]
    for s in PATCH_SIZES
        outcome = filters.outcomes[(s, method, alpha)]
        losses = outcome.lost_gradient ./ max.(image_gradient, eps())
        append!(extreme_ids, sortperm(losses; rev=true)[1:2])
    end
    ids = unique(vcat(sample_ids, extreme_ids))
    name = "q23_keep_drop_m$(method)_alpha$(alpha)"
    figure = comparison_gallery(ids, X, y, patches, name; method=method, alpha=alpha)
    # Lưu đủ sáu gallery; chỉ hiển thị mức α giữa để output notebook gọn.
    alpha == ALPHAS[2] && display(figure)
end

**Cách đọc:** Ở cùng reduction, EdgeLoss thấp hơn biểu thị giữ nhiều tổng magnitude hơn. Bảng cặp gần nhau công bố reduction_gap; đường nối không chứng minh có cấu hình trung gian.

**Câu hỏi gợi ý:** Size nào có đánh đổi tốt hơn ở các điểm gần nhau? Những patch bị loại có chứa đường viền mảnh không?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q2.4 — Filtering có ảnh hưởng khác nhau giữa lớp?

**Quyết định TC:** Phát hiện ứng viên gây mất thông tin không đồng đều.

**Cách tính và đầu vào:** Cùng các metric Q2.2–Q2.3, tổng hợp theo từng lớp 7.000 ảnh.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
evidence_table("q24_class_filter_impact", filters.class_rows; limit=20)
class_spreads = []
for row in filters.global_rows
    matches = filter(r -> r.patch_size == row.patch_size && r.method == row.method && isequal(r.alpha, row.alpha), filters.class_rows)
    losses = collect(skipmissing([r.edge_loss for r in matches]))
    push!(class_spreads, (patch_size=row.patch_size, method=row.method, alpha=row.alpha,
        reduction_range=maximum(r.reduction for r in matches) - minimum(r.reduction for r in matches),
        edge_loss_range=isempty(losses) ? missing : maximum(losses) - minimum(losses),
        max_empty_rate=maximum(r.empty_rate for r in matches)))
end
evidence_table("q24_class_spreads", class_spreads)

In [ ]:
impact_maps = []
for s in PATCH_SIZES, metric in (:reduction, :edge_loss, :empty_rate)
    settings = filter(row -> row.patch_size == s, filters.global_rows)
    matrix = [begin
        row = only(filter(r -> r.patch_size == s && r.label == label && r.method == setting.method && isequal(r.alpha, setting.alpha), filters.class_rows))
        row[metric]
    end for label in 0:9, setting in settings]
    push!(impact_maps, (label="Patch $(s)×$(s), $metric", matrix=matrix))
end
display(evidence_maps(impact_maps, "q24_class_configuration_maps", "Q2.4 Lớp (hàng 0–9) × cấu hình (cột theo q22)"))

**Cách đọc:** Bảng class_spreads thể hiện khoảng chênh giữa lớp; heatmap dùng thứ tự cấu hình trong bảng Q2.2 của từng size.

**Câu hỏi gợi ý:** Lớp nào mất nhiều gradient hơn? Cấu hình nào tạo ảnh hết patch hoặc tác động không đều?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Hàm hỗ trợ — Descriptor raw và histogram hướng cục bộ

In [ ]:
"""Trích raw/255 và histogram hướng 8 bin cho pilot; trả về vectors/metadata/hist và độ nhạy dịch 1px."""
function pilot_descriptors(images, patches, pilot; bins=8)
    result = Dict()
    for size_patch in sort(collect(keys(patches)))
        data = patches[size_patch]
        count_patches = length(data.bounds) * length(pilot)
        raw = Matrix{Float32}(undef, data.area, count_patches)
        histograms = zeros(Float64, bins, count_patches)
        metadata = []
        stability = []
        column = 0
        for id in pilot
            image = @view images[:, :, id]
            fields = sobel_fields(image)
            for bound in data.bounds
                column += 1
                raw[:, column] .= vec(image[bound.r1:bound.r2, bound.c1:bound.c2]) ./ 255
                histogram = orientation_histogram(fields, bound, size(image); bins=bins)
                histograms[:, column] .= histogram
                push!(metadata, (id=id, pid=bound.patch_id))
                if data.gradient[bound.patch_id, id] / data.area >= EDGE_TAU
                    for (dr, dc) in ((1, 0), (0, 1))
                        shifted = orientation_histogram(fields, bound, size(image); bins=bins, dr=dr, dc=dc)
                        denominator = sqrt(sum(abs2, histogram) * sum(abs2, shifted))
                        similarity = denominator == 0 ? missing : sum(histogram .* shifted) / denominator
                        push!(stability, (patch_size=size_patch, image_id=id, patch_id=bound.patch_id,
                            dr=dr, dc=dc, cosine=similarity))
                    end
                end
            end
        end
        result[size_patch] = (raw=raw, histograms=histograms, metadata=metadata, stability=stability)
    end
    return result
end

"""Histogram unsigned [0,π), 8 bin magnitude-weighted; trả về vector; dịch dùng replicate tại biên."""
function orientation_histogram(fields, bound, shape; bins=8, dr=0, dc=0)
    histogram = zeros(Float64, bins)
    for row in bound.r1:bound.r2, column in bound.c1:bound.c2
        rr = clamp(row + dr, 1, shape[1])
        cc = clamp(column + dc, 1, shape[2])
        magnitude = fields.magnitude[rr, cc]
        magnitude == 0 && continue
        angle = mod(atan(fields.gy[rr, cc], fields.gx[rr, cc]), pi)
        bin = min(floor(Int, angle / pi * bins) + 1, bins)
        histogram[bin] += magnitude
    end
    return histogram
end

"""Đếm vector trùng và RMS tới 1024 tham chiếu; bỏ self, trả về nearest/pairs và summary pilot."""
function descriptor_diversity(descriptors, patches, pilot; seed=42, reference_count=1024)
    summaries, pairs, distances, reference_rows = [], [], [], []
    for size_patch in sort(collect(keys(descriptors)))
        descriptor = descriptors[size_patch]
        raw = descriptor.raw
        count_vectors = size(raw, 2)
        rng = MersenneTwister(seed)
        references = sort(randperm(rng, count_vectors)[1:min(reference_count, count_vectors)])
        nearest = fill(Inf, count_vectors)
        matches = zeros(Int, count_vectors)
        for index in 1:count_vectors, reference in references
            index == reference && continue
            distance = 0.0
            for dimension in axes(raw, 1)
                distance += (Float64(raw[dimension, index]) - Float64(raw[dimension, reference]))^2
            end
            distance = sqrt(distance / size(raw, 1))
            if distance < nearest[index]
                nearest[index] = distance
                matches[index] = reference
            end
        end
        for reference in references
            meta = descriptor.metadata[reference]
            push!(reference_rows, (patch_size=size_patch, descriptor_index=reference,
                image_id=meta.id, patch_id=meta.pid))
        end
        for index in 1:count_vectors
            meta = descriptor.metadata[index]
            push!(distances, (patch_size=size_patch, image_id=meta.id, patch_id=meta.pid,
                nearest_reference_rms=nearest[index]))
        end
        data = patches[size_patch]
        constant = count(==(0), data.variance[:, pilot])
        empty = count(==(0), data.foreground[:, pilot, 1])
        push!(summaries, merge((patch_size=size_patch, pilot_images=length(pilot),
            descriptor_dimension=data.area, descriptors=count_vectors,
            references=length(references), unique_exact=length(Set(Tuple(raw[:, i]) for i in 1:count_vectors)),
            constant_fraction=constant / count_vectors, q10_empty_fraction=empty / count_vectors),
            distribution_summary(nearest)))
        for index in sortperm(nearest)[1:min(8, count_vectors)]
            first = descriptor.metadata[index]
            second = descriptor.metadata[matches[index]]
            push!(pairs, (patch_size=size_patch, image_id=first.id, patch_id=first.pid,
                reference_image_id=second.id, reference_patch_id=second.pid, rms=nearest[index]))
        end
    end
    return (summary=summaries, pairs=pairs, distances=distances, references=reference_rows)
end

## I3 — Feature Informativeness: bằng chứng cho descriptor

### Q3.1 — Intensity và gradient thể hiện đặc điểm gì khác nhau?

**Quyết định TC:** Đề xuất raw và gradient descriptor làm ứng viên.

**Cách tính và đầu vào:** Summary toàn tập; scatter chọn tối đa 2.000 patch mỗi size từ pilot bằng seed 42. Các nhóm theo p25/p75 của toàn tập từng size.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
scatter_series = []
for (index, s) in enumerate(PATCH_SIZES)
    data = patches[s]
    variances = vec(data.variance[:, pilot])
    gradients = vec(data.gradient[:, pilot]) ./ data.area
    locations = randperm(MersenneTwister(SEED), length(variances))[1:min(2000, length(variances))]
    push!(scatter_series, (x=Float64.(variances[locations]), y=gradients[locations],
        label="Patch $(s)×$(s), pilot", color=palette[index]))
end
display(evidence_plot(scatter_series, "q31_variance_gradient", "Q3.1 Variance và mean gradient của patch";
    xlabel="Variance intensity²", ylabel="Mean magnitude Sobel / pixel", mode=:scatter))

In [ ]:
patch_groups = select_patch_groups(patches)
evidence_table("q31_patch_group_criteria", patch_groups.groups)
display(patch_example_gallery(patch_groups.examples, X, patches, "q31_patch_group_examples"))

**Cách đọc:** Nhóm variance cao/gradient thấp có thể rỗng; bảng ghi số patch và tiêu chí, không ép chọn ví dụ.

**Câu hỏi gợi ý:** Các vùng đồng nhất sáng chứa thông tin mà gradient bỏ qua không? Hai descriptor thể hiện khác nhau ở những patch nào?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q3.2 — Hướng cạnh đủ đa dạng và ổn định để thử HOG không?

**Quyết định TC:** Đánh giá histogram hướng nhỏ làm ứng viên, chưa khẳng định HOG tốt hơn raw.

**Cách tính và đầu vào:** Histogram unsigned 8 bin magnitude-weighted, chưa block normalization như HOG chuẩn. Patch mean gradient≥80 được kiểm tra cosine khi cửa sổ dịch 1 pixel ngang/dọc; biên replicate.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
descriptors = pilot_descriptors(X, patches, pilot; bins=NBINS)
orientation_rows = []
stability_rows = []
orientation_examples = []
for s in PATCH_SIZES
    descriptor = descriptors[s]
    append!(stability_rows, descriptor.stability)
    for index in axes(descriptor.histograms, 2)
        histogram = descriptor.histograms[:, index]
        energy = sum(histogram)
        probabilities = energy == 0 ? zeros(NBINS) : histogram ./ energy
        entropy = -sum(p * log2(p) for p in probabilities if p > 0; init=0.0)
        meta = descriptor.metadata[index]
        push!(orientation_rows, (patch_size=s, image_id=meta.id, patch_id=meta.pid,
            gradient_energy=energy, nonzero_bins=count(>(0), histogram), entropy_bits=entropy,
            significant=energy / patches[s].area >= EDGE_TAU, dominant_bin=energy == 0 ? 0 : argmax(histogram)))
    end
end
evidence_table("q32_patch_orientation", orientation_rows; limit=10)
if !isempty(stability_rows)
    evidence_table("q32_shift_stability", stability_rows; limit=10)
else
    println("Không có patch pilot đạt mean gradient≥80; chưa đủ bằng chứng đánh giá độ nhạy.")
end

In [ ]:
orientation_series = []
orientation_summaries = []
for (index, s) in enumerate(PATCH_SIZES)
    significant_indices = findall(row -> row.patch_size == s && row.significant, orientation_rows)
    rows = orientation_rows[significant_indices]
    selected = findall(i -> sum(descriptors[s].histograms[:, i]) / patches[s].area >= EDGE_TAU,
        axes(descriptors[s].histograms, 2))
    totals = vec(sum(descriptors[s].histograms[:, selected]; dims=2))
    push!(orientation_series, (x=collect(0:NBINS-1) .* (180 / NBINS), y=totals ./ max(sum(totals), eps()),
        label="Patch $(s)×$(s), mean gradient≥80", color=palette[index]))
    valid_cosines = Float64[r.cosine for r in stability_rows if r.patch_size == s && !ismissing(r.cosine)]
    push!(orientation_summaries, (patch_size=s, significant_patches=length(rows),
        zero_gradient_patches=count(row -> row.patch_size == s && row.gradient_energy == 0, orientation_rows),
        mean_entropy=isempty(rows) ? missing : sum(r.entropy_bits for r in rows) / length(rows),
        mean_shift_cosine=isempty(valid_cosines) ? missing : sum(valid_cosines) / length(valid_cosines)))
end
evidence_table("q32_orientation_summary", orientation_summaries)
display(evidence_plot(orientation_series, "q32_weighted_orientation", "Q3.2 Hướng gradient trên patch có cạnh đáng kể";
    xlabel="Đầu bin hướng (độ), unsigned [0,180)", ylabel="Tỷ trọng tổng magnitude (0–1)"))

In [ ]:
for s in PATCH_SIZES, bin in 1:NBINS
    candidates = filter(row -> row.patch_size == s && row.significant && row.dominant_bin == bin, orientation_rows)
    isempty(candidates) && continue
    row = first(candidates)
    push!(orientation_examples, (size=s, id=row.image_id, pid=row.patch_id,
        group="Hướng trội bin $bin: $((bin - 1) * 180 / NBINS)–$(bin * 180 / NBINS) độ"))
end
display(patch_example_gallery(orientation_examples, X, patches, "q32_orientation_examples"))

**Cách đọc:** Y là tỷ trọng magnitude, entropy dùng đơn vị bit; cosine càng gần 1 càng ít đổi khi dịch cửa sổ. Đây là kiểm tra nhạy vị trí, không phải accuracy của HOG.

**Câu hỏi gợi ý:** Patch 4×4 có quá ít gradient để tạo histogram ổn định không? Hướng trội có khác giữa hai size không?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q3.3 — Descriptor có dư thừa không?

**Quyết định TC:** Thiết kế cách lấy mẫu huấn luyện codebook.

**Cách tính và đầu vào:** Raw/255, không center hoặc chuẩn hóa riêng từng patch. RMS có chia số chiều; tham chiếu 1.024 vector/size seed 42, bỏ self. Cỡ mẫu: 9.800 patch 4×4 và 3.200 patch 7×7.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
diversity = descriptor_diversity(descriptors, patches, pilot; seed=SEED)
evidence_table("q33_raw_descriptor_diversity", diversity.summary)
evidence_table("q33_nearest_reference_pairs", diversity.pairs; limit=16)
evidence_table("q33_reference_manifest", diversity.references; limit=8)
evidence_table("q33_nearest_distances", diversity.distances; limit=8)
display(distribution_plot([[row.nearest_reference_rms for row in diversity.distances if row.patch_size == s] for s in PATCH_SIZES],
    ["Patch $(s)×$(s)" for s in PATCH_SIZES], "q33_nearest_rms_distribution",
    "Q3.3 Khoảng cách RMS đến descriptor tham chiếu gần nhất"; xlabel="RMS trên raw/255"))

pair_examples = []
for row in vcat([filter(r -> r.patch_size == s, diversity.pairs)[1:4] for s in PATCH_SIZES]...)
    push!(pair_examples, (size=row.patch_size, id=row.image_id, pid=row.patch_id, group="Query, RMS=$(round(row.rms; digits=4))"))
    push!(pair_examples, (size=row.patch_size, id=row.reference_image_id, pid=row.reference_patch_id, group="Tham chiếu gần query"))
end
display(patch_example_gallery(pair_examples, X, patches, "q33_similar_patch_pairs"))

In [ ]:
redundancy_rows = []
for s in PATCH_SIZES, method in 0:2, alpha in (method == 0 ? (0.0,) : ALPHAS)
    descriptor = descriptors[s]
    data = patches[s]
    kept = [index for (index, meta) in enumerate(descriptor.metadata)
        if method == 0 || data.foreground[meta.pid, meta.id, method] > floor(Int, alpha * data.area)]
    vectors = Set(Tuple(descriptor.raw[:, index]) for index in kept)
    constant = count(index -> data.variance[descriptor.metadata[index].pid, descriptor.metadata[index].id] == 0, kept)
    push!(redundancy_rows, (patch_size=s, method=method == 0 ? "No-filter" : methods[method],
        alpha=method == 0 ? missing : alpha, pilot_descriptors=length(kept), unique_exact=length(vectors),
        duplicate_fraction=isempty(kept) ? missing : 1 - length(vectors) / length(kept),
        constant_fraction=isempty(kept) ? missing : constant / length(kept)))
end
evidence_table("q33_before_after_filter_redundancy", redundancy_rows)

**Cách đọc:** Nearest chỉ tìm trong tập tham chiếu, không phải toàn bộ descriptor; sample size khác nhau nên tỷ lệ trùng chỉ là bằng chứng pilot.

**Câu hỏi gợi ý:** Lọc có giảm ưu thế của patch đồng nhất không? Cần lấy mẫu codebook cân bằng theo ảnh/vùng hoặc đa dạng descriptor không?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Hàm hỗ trợ — Keypoint và coverage trên một lưới đánh giá cố định

In [ ]:
"""Đổi ảnh Julia row/column sang layout OpenCV; trả về array UInt8 (1,width,height)."""
function opencv_image(image)
    return reshape(permutedims(image), 1, size(image, 2), size(image, 1))
end

"""Chạy một detector trên pilot và ghi counts/tọa độ 0-based; trả về counts và points kể cả ảnh 0 keypoint."""
function extract_keypoints(detector, name, images, labels, pilot)
    counts, points = [], []
    coordinates = Dict{Int, Vector{Tuple{Float64, Float64}}}()
    mask = fill(UInt8(255), 1, size(images, 2), size(images, 1))
    elapsed = @elapsed for id in pilot
        keypoints, descriptors = OpenCV.detectAndCompute(detector, opencv_image(@view(images[:, :, id])), mask)
        descriptor_count = size(descriptors, 3)
        coordinates[id] = Tuple{Float64, Float64}[]
        push!(counts, (method=name, image_id=id, label=labels[id], keypoints=length(keypoints),
            descriptors=descriptor_count, dimension=descriptor_count == 0 ? 0 : size(descriptors, 2)))
        for (index, point) in enumerate(keypoints)
            x, yy = Float64(point.pt.x), Float64(point.pt.y)
            push!(coordinates[id], (x, yy))
            push!(points, (method=name, image_id=id, point_id=index, x_zero_based=x, y_zero_based=yy))
        end
    end
    return (counts=counts, points=points, coordinates=coordinates, seconds=elapsed)
end

"""Đo coverage fixed grid 7×7 ô 4×4; mẫu số rỗng -> missing; trả về bảng cấp ảnh/phương pháp/ngưỡng."""
function keypoint_coverage(extracted, patches, labels, thresholds, images)
    rows = []
    target = patches[4]
    columns = size(images, 2) ÷ 4
    for extracted_method in extracted
        name = first(extracted_method.counts).method
        for id in sort(collect(keys(extracted_method.coordinates)))
            points = extracted_method.coordinates[id]
            hit = falses(length(target.bounds))
            for (x, yy) in points
                column = clamp(floor(Int, x) + 1, 1, size(images, 2))
                row = clamp(floor(Int, yy) + 1, 1, size(images, 1))
                hit[((row - 1) ÷ 4) * columns + (column - 1) ÷ 4 + 1] = true
            end
            edge_targets = target.gradient[:, id] ./ target.area .>= EDGE_TAU
            for method in 1:2
                foreground_targets = target.foreground[:, id, method] .> 0
                on_foreground = count(points) do (x, yy)
                    column = clamp(floor(Int, x) + 1, 1, size(images, 2))
                    row = clamp(floor(Int, yy) + 1, 1, size(images, 1))
                    images[row, column, id] > thresholds[method]
                end
                push!(rows, (extractor=name, image_id=id, label=labels[id], threshold_method=methods[method],
                    keypoints=length(points), foreground_targets=count(foreground_targets), edge_targets=count(edge_targets),
                    on_foreground=isempty(points) ? missing : on_foreground / length(points),
                    foreground_patch_coverage=any(foreground_targets) ? count(hit .& foreground_targets) / count(foreground_targets) : missing,
                    edge_patch_coverage=any(edge_targets) ? count(hit .& edge_targets) / count(edge_targets) : missing))
            end
        end
    end
    return rows
end

"""Vẽ pilot đại diện với keypoints và mask/gradient gốc; lấy ids thuộc pilot; trả về SVG."""
function keypoint_gallery(ids, extracted, images, thresholds, name)
    io = IOBuffer()
    for (index, id) in enumerate(ids)
        yy = 48 + (index - 1) * 125
        image = @view images[:, :, id]
        svg_text(io, 12, yy, "ID $id | lớp $(y[id])"; size=11)
        for (method, threshold) in enumerate(thresholds)
            x = 15 + (method - 1) * 105
            draw_image(io, image .> threshold, x, yy + 12; scale=3, binary=true)
            svg_text(io, x, yy + 108, methods[method]; size=9)
        end
        magnitude = sobel_fields(image).magnitude
        gradient_image = round.(UInt8, clamp.(magnitude ./ (4 * 255 * sqrt(2)), 0, 1) .* 255)
        draw_image(io, gradient_image, 225, yy + 12; scale=3)
        svg_text(io, 225, yy + 108, "Gradient gốc"; size=9)
        for (method_index, method) in enumerate(extracted)
            x = 330 + (method_index - 1) * 110
            draw_image(io, image, x, yy + 12; scale=3)
            for (px, py) in method.coordinates[id]
                println(io, "<circle cx=\"$(x + (px + 0.5) * 3)\" cy=\"$(yy + 12 + (py + 0.5) * 3)\" r=\"3\" fill=\"none\" stroke=\"#e11d48\"/>")
            end
            svg_text(io, x, yy + 108, first(method.counts).method; size=9)
        end
    end
    return svg_save(joinpath(OUT, name * ".svg"), "Pilot: mask, gradient và keypoints", String(take!(io)), 610, 52 + length(ids) * 125)
end

## I4 — Feature Extractability: dense hay keypoint-based?

### Q4.1 — SIFT/ORB tạo đủ keypoints không?

**Quyết định TC:** Quyết định có đưa phương án keypoint-based vào TC thử nghiệm không.

**Cách tính và đầu vào:** Giữ cấu hình extractor hiện tại, pilot cân bằng 200 ảnh; ảnh 0 descriptor vẫn nằm trong mẫu số.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
orb_parameters = (nfeatures=500, scaleFactor=1.2, nlevels=1, edgeThreshold=8,
    firstLevel=0, WTA_K=2, patchSize=15, fastThreshold=10)
sift_parameters = (nfeatures=0, nOctaveLayers=3, contrastThreshold=0.04,
    edgeThreshold=10.0, sigma=1.6, enable_precise_upscale=false)
orb = OpenCV.ORB_create(; orb_parameters...)
sift = OpenCV.SIFT_create(; sift_parameters...)
println("ORB: ", orb_parameters)
println("SIFT: ", sift_parameters)

In [ ]:
orb_result = extract_keypoints(orb, "ORB", X, y, pilot)
println("ORB pilot: ", orb_result.seconds, " giây")

In [ ]:
sift_result = extract_keypoints(sift, "SIFT", X, y, pilot)
println("SIFT pilot: ", sift_result.seconds, " giây")

In [ ]:
extracted = [orb_result, sift_result]
keypoint_counts = vcat([result.counts for result in extracted]...)
keypoint_points = vcat([result.points for result in extracted]...)
evidence_table("q41_keypoint_counts", keypoint_counts; limit=10)
if !isempty(keypoint_points)
    evidence_table("q41_keypoint_coordinates", keypoint_points; limit=10)
else
    write_csv(joinpath(OUT, "q41_keypoint_coordinates.csv"),
        ["method", "image_id", "point_id", "x_zero_based", "y_zero_based"], [])
end
keypoint_summary = [merge((method=first(result.counts).method, pilot_images=length(pilot),
    zero_rate=count(row -> row.keypoints == 0, result.counts) / length(pilot),
    zero_descriptor_rate=count(row -> row.descriptors == 0, result.counts) / length(pilot),
    low_descriptor_rate=count(row -> row.descriptors <= 2, result.counts) / length(pilot),
    pilot_seconds=result.seconds), distribution_summary([row.keypoints for row in result.counts])) for result in extracted]
evidence_table("q41_extractor_summary", keypoint_summary)

In [ ]:
keypoint_classes = []
for result in extracted, label in 0:9
    rows = filter(row -> row.label == label, result.counts)
    push!(keypoint_classes, merge((method=first(rows).method, label=label,
        zero_rate=count(row -> row.keypoints == 0, rows) / length(rows),
        low_descriptor_rate=count(row -> row.descriptors <= 2, rows) / length(rows)),
        distribution_summary([row.keypoints for row in rows])))
end
evidence_table("q41_keypoints_by_class", keypoint_classes)
display(distribution_plot([[row.keypoints for row in result.counts] for result in extracted],
    ["ORB", "SIFT"], "q41_keypoint_histogram", "Q4.1 Số keypoint mỗi ảnh pilot";
    xlabel="Số keypoint / ảnh", bins=24))

**Cách đọc:** Kết quả chỉ trên pilot; zero-rate và low-rate cùng mẫu số 200, mỗi lớp 20 ảnh.

**Câu hỏi gợi ý:** Phương pháp nào thường không tạo descriptor? Có lớp nào chịu zero-rate cao hơn không?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q4.2 — Keypoints bao phủ vùng có thông tin không?

**Quyết định TC:** So sánh tính khả dụng của dense patches và keypoint-based.

**Cách tính và đầu vào:** Coverage trên cùng lưới 7×7 ô 4×4, không thay theo kích thước descriptor. Target foreground: count>0; target cạnh: mean magnitude≥80. Tọa độ OpenCV 0-based chuyển về Julia 1-based.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
coverage_rows = keypoint_coverage(extracted, patches, y, taus, X)
evidence_table("q42_coverage_per_image", coverage_rows; limit=10)
coverage_summary = []
for extractor in ("ORB", "SIFT"), method in methods, label in -1:9
    rows = filter(row -> row.extractor == extractor && row.threshold_method == method && (label == -1 || row.label == label), coverage_rows)
    for metric in (:on_foreground, :foreground_patch_coverage, :edge_patch_coverage)
        values = Float64[row[metric] for row in rows if !ismissing(row[metric])]
        @assert all(value -> 0 <= value <= 1, values)
        push!(coverage_summary, (extractor=extractor, threshold_method=method, label=label, metric=string(metric),
            valid_images=length(values), missing_images=length(rows) - length(values),
            mean=isempty(values) ? missing : sum(values) / length(values)))
    end
end
evidence_table("q42_coverage_summary", coverage_summary; limit=18)

In [ ]:
keypoint_gallery_ids = [first(filter(id -> y[id] == label, pilot)) for label in 0:9]
display(keypoint_gallery(keypoint_gallery_ids, extracted, X, taus, "q42_keypoint_mask_gradient_gallery"))

**Cách đọc:** Hit chỉ cần một keypoint có tâm nằm trong ô. Đây là coverage patch mục tiêu, không phải phần trăm pixel được descriptor mô tả. Ảnh không keypoint có coverage 0 khi có target; mẫu số 0 là missing.

**Câu hỏi gợi ý:** Keypoints bỏ sót vùng nào so với dense grid? Chúng tập trung vào foreground hay chỉ vài vị trí?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Hàm hỗ trợ — Vùng không gian và ánh xạ patch

In [ ]:
"""Phân vùng theo tọa độ tâm 1-based; trả về tên vùng; ranh giới 14.5 được gán xuống/phải."""
function spatial_region(row, column, scheme)
    if scheme == "bands3"
        return row <= 9 ? "top" : row <= 19 ? "middle" : "bottom"
    end
    return (row < 14.5 ? "top-" : "bottom-") * (column < 14.5 ? "left" : "right")
end

"""Tổng hợp pixel theo cùng vùng và class; không phụ thuộc size patch; trả về bảng regional."""
function regional_statistics(spatial)
    rows = []
    for label in -1:9, scheme in ("bands3", "quadrants")
        mean_image = label == -1 ? spatial.global_mean : spatial.class_mean[:, :, label + 1]
        gradient = label == -1 ? spatial.global_gradient : spatial.class_gradient[:, :, label + 1]
        regions = unique(spatial_region(row, column, scheme) for row in 1:28 for column in 1:28)
        for region in regions
            indices = [CartesianIndex(row, column) for row in 1:28 for column in 1:28
                if spatial_region(row, column, scheme) == region]
            for method in 1:2
                occupancy = label == -1 ? spatial.global_occupancy[:, :, method] : spatial.class_occupancy[:, :, label + 1, method]
                push!(rows, (label=label, scheme=scheme, region=region, method=methods[method], pixels=length(indices),
                    mean_intensity=sum(mean_image[indices]) / length(indices),
                    foreground_occupancy=sum(occupancy[indices]) / length(indices),
                    mean_gradient=sum(gradient[indices]) / length(indices),
                    gradient_share=sum(gradient) == 0 ? missing : sum(gradient[indices]) / sum(gradient)))
            end
        end
    end
    return rows
end

"""Ánh xạ patch nguyên vẹn bằng tâm vào vùng; trả về bảng mapping, không tạo visual words."""
function spatial_patch_mapping(patches)
    rows = []
    for size_patch in sort(collect(keys(patches))), scheme in ("bands3", "quadrants")
        for bound in patches[size_patch].bounds
            center_row = (bound.r1 + bound.r2) / 2
            center_column = (bound.c1 + bound.c2) / 2
            push!(rows, (patch_size=size_patch, patch_id=bound.patch_id, scheme=scheme,
                center_row=center_row, center_column=center_column,
                region=spatial_region(center_row, center_column, scheme), represented_pixels=size_patch^2))
        end
    end
    return rows
end

## I5 — Spatial Relevance: có cơ sở giữ vị trí không?

### Q5.1 — Đặc trưng có phụ thuộc vị trí?

**Quyết định TC:** Đề xuất các vùng spatial đơn giản trước encoding.

**Cách tính và đầu vào:** Thống kê pixel trên cùng hệ tọa độ ảnh: top rows1:9, middle10:19, bottom20:28; quadrants rows/cols1:14 và15:28. Báo diện tích và gradient share để không nhầm vùng lớn với nhiều chi tiết hơn.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
regional_rows = regional_statistics(spatial)
evidence_table("q51_regional_statistics", regional_rows; limit=14)
global_regions = filter(row -> row.label == -1 && row.method == methods[1], regional_rows)
regional_series = [(x=collect(1:length(global_regions)), y=[r.mean_gradient for r in global_regions],
    label="Thứ tự vùng: " * join([r.region for r in global_regions], ", "), color=palette[1])]
display(evidence_plot(regional_series, "q51_region_mean_gradient", "Q5.1 Mean gradient ở các vùng không gian";
    xlabel="STT vùng theo bảng global, Q10+", ylabel="Mean magnitude / pixel"))

In [ ]:
region_distribution_rows = evidence.regional_per_image
write_csv(joinpath(CACHE, "regional_per_image.csv"), string.(keys(first(region_distribution_rows))),
    (Tuple(row) for row in region_distribution_rows))
println("Phân phối vùng tính cùng lượt Sobel: ", length(region_distribution_rows), " dòng ngoài Git")

In [ ]:
regional_distribution_summary = []
for scheme in ("bands3", "quadrants")
    names = unique(row.region for row in region_distribution_rows if row.scheme == scheme)
    for name in names, metric in (:mean_intensity, :mean_gradient, :fr_q10, :fr_otsu)
        values = [row[metric] for row in region_distribution_rows if row.scheme == scheme && row.region == name]
        push!(regional_distribution_summary, merge((scheme=scheme, region=name, metric=string(metric)), distribution_summary(values)))
    end
end
evidence_table("q51_region_distribution_summary", regional_distribution_summary; limit=16)
for metric in (:mean_intensity, :mean_gradient, :fr_q10, :fr_otsu)
    names = ("top", "middle", "bottom")
    values = [[row[metric] for row in region_distribution_rows if row.scheme == "bands3" && row.region == name] for name in names]
    display(distribution_plot(values, collect(names), "q51_bands_$(metric)", "Q5.1 Phân phối $metric theo dải ngang";
        xlabel=string(metric)))
end

**Cách đọc:** Mean gradient/pixel và gradient share có mẫu số khác nhau; middle có 10 hàng, top/bottom có 9 hàng.

**Câu hỏi gợi ý:** Vùng nào có đặc trưng khác nhau? Chênh lệch có còn sau khi chia theo diện tích không?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q5.2 — Spatial patterns có khác nhau giữa lớp?

**Quyết định TC:** Xem vị trí có mang thông tin hình dạng đáng thử cho TC6.

**Cách tính và đầu vào:** Mean, variance, gradient và occupancy trên ảnh gốc, cùng thang màu cho mỗi loại metric.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
for (name, title, global_matrix, class_matrices) in (
    ("mean", "Mean intensity", spatial.global_mean, spatial.class_mean),
    ("variance", "Variance intensity", spatial.global_variance, spatial.class_variance),
    ("gradient", "Mean gradient", spatial.global_gradient, spatial.class_gradient))
    entries = vcat([(label="Toàn tập", matrix=global_matrix)],
        [(label="Lớp $label — $(CLASSES[label + 1])", matrix=class_matrices[:, :, label + 1]) for label in 0:9])
    display(evidence_maps(entries, "q52_$(name)_maps", "Q5.2 $title theo vị trí pixel"))
end

In [ ]:
for method in 1:2
    entries = vcat([(label="Toàn tập", matrix=spatial.global_occupancy[:, :, method])],
        [(label="Lớp $label — $(CLASSES[label + 1])", matrix=spatial.class_occupancy[:, :, label + 1, method]) for label in 0:9])
    display(evidence_maps(entries, "q52_occupancy_m$(method)", "Q5.2 Foreground occupancy — $(methods[method])"))
    @assert isapprox(sum(spatial.global_occupancy[:, :, method]) / NPX, sum(fr[:, method]) / N; atol=1e-12)
end

**Cách đọc:** Mỗi pixel là thống kê của 7.000 ảnh/lớp; màu giữa lớp so sánh trực tiếp trong cùng một hình.

**Câu hỏi gợi ý:** Các lớp có vùng trên/giữa/dưới khác nhau rõ rệt không? Spatial words có thể bổ sung ý nghĩa gì cần kiểm chứng?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

### Q5.3 — Nên thử cách chia vùng nào?

**Quyết định TC:** Bàn giao quy tắc spatial encoding rõ ràng cho TC6.

**Cách tính và đầu vào:** Gán nguyên patch theo tâm. Vùng pixel và vùng đại diện bởi tâm patch có thể khác ở biên; báo số patch/diện tích mỗi vùng.

**Kết quả mong đợi:** bảng/hình có số liệu, đơn vị và cấu hình rõ ràng; không tự chốt cấu hình tối ưu.

In [ ]:
spatial_mapping = spatial_patch_mapping(patches)
evidence_table("q53_patch_region_mapping", spatial_mapping; limit=14)
spatial_sizes = []
for s in PATCH_SIZES, scheme in ("bands3", "quadrants")
    rows = filter(row -> row.patch_size == s && row.scheme == scheme, spatial_mapping)
    for region in unique(row.region for row in rows)
        matches = filter(row -> row.region == region, rows)
        push!(spatial_sizes, (patch_size=s, scheme=scheme, region=region,
            patch_count=length(matches), represented_pixel_slots=sum(row.represented_pixels for row in matches)))
    end
end
evidence_table("q53_region_patch_counts", spatial_sizes)
evidence_table("q53_spatial_candidates", [
    (candidate="No-spatial", item_universe="K", rule="Visual word chỉ có ID"),
    (candidate="bands3", item_universe="3K", rule="Tâm hàng≤9: top; ≤19: middle; còn lại bottom"),
    (candidate="quadrants", item_universe="4K", rule="Tâm hàng/cột<14.5: trên/trái; bằng ranh giới: dưới/phải"),
])

In [ ]:
mapping_entries = []
for s in PATCH_SIZES, scheme in ("bands3", "quadrants")
    rows = filter(row -> row.patch_size == s && row.scheme == scheme, spatial_mapping)
    names = sort(unique(row.region for row in rows))
    side = H ÷ s
    matrix = zeros(Int, side, side)
    for row in rows
        bound = patches[s].bounds[row.patch_id]
        matrix[bound.patch_row, bound.patch_col] = findfirst(==(row.region), names)
    end
    push!(mapping_entries, (label="$(s)×$(s), $scheme: " * join(names, "/"), matrix=matrix))
end
display(evidence_maps(mapping_entries, "q53_patch_region_maps", "Q5.3 Vùng theo tâm patch — mã màu theo tên vùng"))

**Cách đọc:** K là số visual words sẽ thử ở TC; K/3K/4K là giới hạn item universe, chưa có support. Mỗi scheme đánh số vùng theo tên hiển thị, không so màu giữa scheme.

**Câu hỏi gợi ý:** Có vùng nào nhận quá nhiều/ít patch? Cần kiểm chứng transaction length và support như thế nào tại EDA-2?

**Insight của tôi:** _Ghi nhận xét cùng số liệu, image_id/patch_id và kích thước patch._

**Cần kiểm chứng tiếp:** _Ghi giới hạn bằng chứng và thử nghiệm cần làm tại TC/EDA-2._

## R — Bàn giao bằng chứng cho TC

Không tự tuyên bố patch nào tốt hơn. Quyết định cần dựa đồng thời vào cấu trúc patch, reduction–loss, mức ảnh hưởng theo lớp và descriptor volume. Nếu chưa phân biệt rõ, mang cả hai kích thước sang TC để đối chứng.

In [ ]:
decision_rows = [
    (component="TC1 — Patch extraction", question="4×4 hay 7×7 làm baseline?", evidence="q11/q12/q13; i1 gallery; q23 tradeoff", observation="", interpretation="", candidate="", validation_needed="So codebook và transactions EDA-2"),
    (component="TC2 — Filtering", question="No-filter hay τ/α nào?", evidence="q21/q22/q23/q24", observation="", interpretation="", candidate="", validation_needed="Không bỏ ảnh hết patch; đo chi phí và support"),
    (component="TC3 — Descriptor", question="Raw/gradient/histogram hướng nào đáng thử?", evidence="q31/q32/q33", observation="", interpretation="", candidate="", validation_needed="Thiết kế normalization, K-means và chất lượng words"),
    (component="TC3 — Strategy", question="Dense hay SIFT/ORB?", evidence="q41/q42; pilot 200 ảnh", observation="", interpretation="", candidate="", validation_needed="Kiểm tra quy mô toàn tập và ảnh zero-feature"),
    (component="TC6 — Encoding", question="No-spatial hay bands3/quadrants?", evidence="q51/q52/q53", observation="", interpretation="", candidate="", validation_needed="Đo item count, transaction length, support và ý nghĩa patterns"),
]
# Không ghi đè các lựa chọn người dùng đã điền ở lần chạy trước.
decision_path = joinpath(OUT, "tc_decision_template.csv")
if !isfile(decision_path)
    evidence_table("tc_decision_template", decision_rows)
else
    println("Giữ bản quyết định người dùng đã có: ", relpath(decision_path, repo_root))
end

### R02 — Báo cáo dữ kiện và giới hạn bằng chứng

In [ ]:
report_path = joinpath(OUT, "eda1_tc_evidence_report.md")
open(report_path, "w") do io
    println(io, "# EDA-1 — Bằng chứng so sánh patch cho TC\n")
    println(io, "- $N ảnh; index SHA-256: $idxsha.")
    println(io, "- Q10+=$q10; Otsu=$otsu; mask I>τ; α=$(collect(ALPHAS)).")
    println(io, "- Patch 4×4/7×7 giữ đủ 784 pixel trước lọc; metrics $(metrics_rows) dòng.")
    println(io, "- Pilot $(length(pilot)) ảnh, $(PILOT_PER_CLASS)/lớp, seed $SEED.")
    println(io, "\n## Dữ kiện filtering\n")
    for row in filters.global_rows
        println(io, "- $(row.patch_size)×$(row.patch_size), $(row.method), α=$(row.alpha): reduction=$(row.reduction), EdgeLoss=$(row.edge_loss), empty_rate=$(row.empty_rate), retained_pixel_slots=$(row.retained_pixel_slots).")
    end
    println(io, "\n## Chỗ ghi kết luận của người phân tích\n")
    println(io, "Điền tc_decision_template.csv, đối chiếu Q1.1–Q5.3 và ghi số liệu/ID cụ thể.")
    println(io, "\n## Giới hạn\n")
    println(io, "EdgeLoss là proxy tổng magnitude, không chứng minh chi tiết ngữ nghĩa. Khoảng cách descriptor và extractability chỉ trên pilot. Chưa có codebook/visual words, support hoặc kết quả Eclat; chưa kết luận size/descriptor/spatial tốt nhất.")
end
display(Markdown.parse(read(report_path, String)))

### R03 — Metadata phiên chạy và output kiểm tra bàn giao

In [ ]:
run_metadata = merge(provenance, Dict(
    "pilot_images" => length(pilot), "pilot_per_class" => PILOT_PER_CLASS,
    "orientation_bins" => NBINS, "edge_mean_threshold" => EDGE_TAU,
    "raw_normalization" => "Float32(pixel)/255; no per-patch normalization",
    "reference_count" => 1024, "near_zero_variance_rule" => "0 < variance <= 1",
    "orb_parameters" => Dict(string(k) => v for (k, v) in pairs(orb_parameters)),
    "sift_parameters" => Dict(string(k) => v for (k, v) in pairs(sift_parameters)),
    "input_path" => relpath(DATA, repo_root), "output_path" => relpath(OUT, repo_root),
))
open(joinpath(OUT, "run_metadata.toml"), "w") do io
    TOML.print(io, run_metadata; sorted=true)
end

output_rows = [(relative_path=relpath(path, repo_root), bytes=filesize(path),
    sha256=bytes2hex(open(sha256, path))) for path in sort(readdir(OUT; join=true))
    if isfile(path) && basename(path) != "output_manifest.csv"]
evidence_table("output_manifest", output_rows; limit=16)

### R04 — Nghiệm thu và insight của tôi

- Chạy notebook từ kernel Julia mới; xác nhận không có error và các assertion đều qua.
- Kiểm tra gallery đúng hướng, đủ lớp, không cắt chữ, ví dụ giữ/loại đúng cutoff.
- Viết insight và lựa chọn TC vào Markdown/`tc_decision_template.csv`, kèm bằng chứng.
- Chỉ tick **đã chạy/có bằng chứng** trong checklist sau khi đối chiếu output của phiên này. Code mới được viết và kiểm tra cú pháp không đồng nghĩa đã có kết quả trên 70.000 ảnh.
- Không sử dụng output cũ ở thư mục cha làm bằng chứng cho bộ α mới.